# CLDT-Thread — Phase 6: Evidence Freeze and Reproducible Handoff
## Final Repetitions, Automated Reproduction, Limitations, dan Presentation

**Repository snapshot:** `fcfa1837730663ce617b6c65ce11c13d6e248499`  
**Planning window:** 20–26 October 2026, sesuai Week 6 pada calendar repository  
**Handoff boundary:** 27 October 2026; tidak ada tuning, feature work, atau penggabungan evidence setelah batas ini  
**Notebook status:** implementation workbook; bukan bukti bahwa perangkat, analisis, atau klaim sudah berhasil

Phase 6 bukan minggu untuk menambah kemampuan. Minggu ini mengunci identitas implementasi yang telah lolos gerbang Phase 1–5, menyelesaikan repetitions yang sudah dideklarasikan, menjalankan `host/analysis/reproduce.py` dari raw evidence, mencatat limitation dan hasil negatif tanpa menyembunyikannya, lalu menguji apakah orang kedua dapat mereproduksi hasil dari clean clone.

| Outcome yang sah | Maknanya |
|---|---|
| Positive | Bukti lengkap dan frozen analysis mendukung klaim yang sudah dideklarasikan |
| Negative | Bukti lengkap menunjukkan model atau safety case tidak memenuhi acceptance; ini tetap hasil ilmiah |
| Inconclusive | Bukti lengkap, tetapi uncertainty atau jumlah valid run tidak cukup untuk keputusan |
| Non-evaluable | Evidence chain, identity, atau reconciliation rusak; tidak boleh diubah menjadi klaim |
| Remote-off | Default untuk semua safety invariant yang belum terbukti, termasuk ketika result negative, inconclusive, atau non-evaluable |


## Week-6 Calendar

![Week-6 implementation roadmap](docs/diagrams/phase6/implementation-roadmap.svg)

Proporsi calendar mempertahankan pekerjaan serial yang tidak dapat dipadatkan: final physical repetitions selesai sebelum result freeze, sedangkan reconciliation berjalan segera setelah setiap run. `topology-shift` dan ablation tidak memperoleh slot wajib. Keduanya hanya boleh memakai waktu tersisa bila sudah stabil pada 19 October dan seluruh core evidence sudah lengkap.


## Notebook Boundary
### Exact Source Copies, Bukan Repository Adapter

Notebook ini berfungsi sebagai panduan dan tempat membaca source. Cell source menyalin file repository secara persis ke `/content/cldt_scratch` bila sengaja dijalankan; tidak ada cell yang menulis balik ke repository, mengganti TODO, menghasilkan angka contoh sebagai hasil, atau menyatakan hardware sudah lulus. Implementasi nyata tetap dilakukan pada path repository yang disebut pada setiap bagian.

Nilai contoh hanya menjelaskan **bentuk isian**, bukan data proyek. Contoh `pass` berarti kolom verdict membutuhkan vocabulary yang dibekukan; contoh `100` berarti kolom tersebut membutuhkan angka hasil ukur dengan unit yang disebut. Nilai final hanya berasal dari manifest frozen, perangkat, log, atau output reproduction.


In [ ]:
from pathlib import Path
Path("/content/cldt_scratch").mkdir(parents=True, exist_ok=True)


## 1. Scope Freeze dan Definition of Done

Core Week 6 terdiri dari empat jalur yang saling mengunci:

1. Menutup contract `host/analysis/reproduce.py` dan membuatnya gagal keras pada evidence yang malformed, tidak lengkap, atau tidak reconciled.
2. Menjalankan final set yang sudah eligible: `stable-baseline` dan `load-step-prediction`; `stale-observation-fallback` serta `restart-replay-safety` hanya bila gerbang Week 5 sudah lengkap.
3. Membekukan raw evidence, manifest digest, source/binary identity, run ledger, analysis configuration, dan hasil turunan sebagai satu evidence chain.
4. Menyusun limitation dan presentation dari chain tersebut tanpa mengubah acceptance, split, feature set, exclusion, atau repetition count setelah outcome terlihat.

Tidak termasuk Week 6: fitur C/C++/firmware baru, node baru, sensor, dashboard, SMP, power measurement, SPI migration, multi-action control, atau pembelian hardware. Perbaikan source hanya sah bila benar-benar merupakan **evidence defect**. Setiap perubahan behavior setelah satu final run memulai identity block baru; run sebelum dan sesudah perubahan tidak digabung.

### Frozen Phase-1–5 Owners yang Menjadi Dependency

Week 6 tidak mengerjakan ulang owner berikut, tetapi preflight dan reproduction harus membuktikan bahwa binary/evidence yang dipakai memang berasal dari implementasinya: `host/experiment_config.h/.c`, `host/coordinator.h/.c`, `host/main.c`, `common/include/cldt/cldt_status.h`, `common/include/cldt/cldt_types.h`, `common/include/cldt/cldt_protocol.h`, `common/src/cldt/cldt_protocol.c`, `common/include/cldt/cldt_metrics.h`, `common/src/cldt/cldt_metrics.c`, `common/include/cldt/cldt_event_trace.h`, `common/src/cldt/cldt_event_trace.c`, `host/twin_model.h/.c`, `host/estimator.h/.c`, `host/kalman.h/.c`, `host/fidelity_gate.h/.c`, dan `host/policy.h/.c`. Gateway/endpoint source, provisioning, transport, replay store, local apply, build configuration, serta entry points tetap mengikuti PHASE1–5.

Owner tersebut dibuka kembali hanya ketika gate atau evidence menunjukkan defect pada contract yang dimilikinya. Perubahan sekecil apa pun yang mengubah frame, counter, model, decision, command, fallback, atau replay semantics memerlukan build/test ulang dan identity rollover; Week-6 runs lama tidak dicampur dengan runs setelah repair.


## 2. Entry Gate dari Phase 1–5

Phase 6 dimulai dari gate, bukan dari tanggal. Tabel ini merupakan decision record yang diisi dari evidence nyata.

| Gate | Evidence yang dirujuk | Bentuk verdict | Konsekuensi bila belum lulus |
|---|---|---|---|
| Phase 1 | Build, fixed vectors, pinned toolchain, upstream images | contoh `pass` / `fail` | Hentikan final campaign; hanya foundation repair |
| Phase 2 | Local accounting, cold boot, one-endpoint soak, second endpoint | contoh `pass` / `fail` | Batasi deliverable ke local/physical baseline yang benar-benar bekerja |
| Phase 3 | Project frames, raw recorder, lifecycle audit, aggregate reconciliation | contoh `pass` / `fail` | Tidak ada model claim |
| Phase 4 | Frozen calibration/held-out split dan three-model shadow score | contoh `positive` / `negative` / `inconclusive` / `non-evaluable` | Negative boleh dilaporkan; selain positive tidak membuka actuation |
| Phase 5 | Normal finite action, stale fallback/requalification, restart/replay rejection | contoh `pass` / `fail` / `not eligible` | Pertahankan remote-off; safety claim hanya untuk invariant yang terbukti |

Untuk setiap row, catatan mencantumkan evidence-bundle identity, manifest digest, source revision, binary hashes, operator, tanggal, dan alasan. Label `pass` tanpa rujukan evidence tidak dihitung sebagai gate.


### Cut Rules sebelum Final Runs

1. Jika raw lifecycle belum reconciled, run diklasifikasikan invalid; analysis tidak boleh menghapus item bermasalah.
2. Jika source, binary, topology, channel, placement, control profile, key identity, atau model revision berubah, block lama ditutup dan block baru diberi identity baru.
3. Jika Week-4 result bukan frozen positive, `remote_actuation` tetap `false` dan Week-5 physical actuation tidak diulang sebagai jalan pintas.
4. Jika stale fallback atau replay rejection gagal, tidak ada usaha “mendapat run bagus”; failure dipertahankan sebagai evidence, defect diperbaiki, lalu campaign baru memakai identity baru.
5. Jika core queue belum selesai pada 24 October, topology shift, ablation, kosmetik, dan perluasan dokumentasi dipotong lebih dulu.


## 3. Freeze Record sebelum Outcome Terlihat

Satu freeze record mengikat keputusan yang berpotensi mengubah hasil. Label di bawah adalah metadata manusia atau field repository yang sudah ada; notebook tidak memperkenalkan variabel runtime baru.

| Keputusan | Isi dari repo/evidence | Contoh bentuk, bukan nilai proyek |
|---|---|---|
| Source revision | Git commit penuh | `fcfa…` |
| Manifest identity | `experiment_id` + SHA-256 strict JSON | `stable-baseline / <64 hex>` |
| Physical identity | Empat label board, role, serial, port, placement, channel | `endpoint-a / child / …` |
| Binary identity | SHA-256 gateway, RCP, endpoint A/B | `<64 hex>` |
| Toolchain identity | ESP-IDF/upstream/CMake/compiler revision | `vX.Y / commit` |
| Analysis identity | `reproduce.py` revision dan dependency versions | `commit / package version` |
| Execution plan | `warmup_s`, `measurement_s`, `cooldown_s`, `repetitions`, `seed` | angka terukur/frozen |
| Statistical plan | calibration block, held-out block, horizon, metric, uncertainty | named evidence blocks |
| Acceptance | reconciliation, service floor, negative case | `true` / angka 0–1 / teks frozen |
| Eligibility | baseline/load/stale/restart/topology status | `eligible` / `not eligible: reason` |

Repetition count tidak ditebak dari schema maksimum. Minimum tiga pilot non-reportable digunakan untuk memahami variance dan durasi; jumlah final dipilih sebelum treatment outcome dibaca. Pilot, invalid, interrupted, dan final runs memiliki status terpisah dan semuanya tetap tercatat.


## 4. Reproduction Contract yang Harus Ditutup

`host/analysis/reproduce.py` adalah satu-satunya direct Week-6 implementation owner. Source saat ini menyebut satu `<results_dir>`, sedangkan pertanyaan ilmiah membutuhkan banyak physical runs, calibration block, held-out block, dan paired horizons. Sebelum body TODO ditulis, contract berikut perlu dibekukan dalam dokumentasi kode atau CLI help:

1. Apakah satu invocation menerima satu run directory atau satu collection directory; pilih satu dan jangan membuat script menebak struktur.
2. Bagaimana `execution.repetitions` dipetakan ke run identity dan directory; satu run tidak boleh diam-diam mewakili beberapa repetitions.
3. Exact record contract `events.ndjson`: required keys, type, unit, vocabulary, payload encoding, ordering expectation, dan maximum size.
4. Sumber collection-level binding untuk baseline/calibration/held-out serta allowed experiment IDs.
5. Nama, lokasi, format, dan overwrite policy output; output tidak boleh masuk ke raw directory.
6. Dependency versions untuk NumPy, SciPy, dan Matplotlib; versi yang menjalankan final analysis harus dapat dipasang ulang.
7. Exit-code contract untuk usage error, missing evidence, digest mismatch, malformed NDJSON, reconciliation failure, dan analysis failure.
8. Hubungan interval prediksi dengan contract Phase 4. TODO `+/- 2 sigma` tidak boleh mengganti interval frozen dengan aturan baru.
9. Duplicate definition: raw duplicate reception dan duplicate lifecycle event perlu dibedakan secara eksplisit.
10. Seed bootstrap dan semua stochastic operation dibekukan serta direkam.

Tanpa keputusan ini, mengisi function body akan menghasilkan output yang tampak rapi tetapi tidak reproducible.


### `host/analysis/reproduce.py`

Cell berikut adalah salinan persis dari `host/analysis/reproduce.py` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/reproduce.py
import sys
import json
from pathlib import Path

def main():
    if len(sys.argv) != 2:
        print("Usage: python reproduce.py <results_dir>")
        sys.exit(1)
        
    # TODO: load manifest JSON from results_dir / "manifest.json"
    # TODO: verify manifest has state="ready" and all _todo items resolved
    # TODO: compute SHA-256 digest of manifest and compare against results_dir / "manifest.sha256"
    # TODO: load events.ndjson one JSON object per line; reject malformed, blank,
    # duplicate, or trailing non-JSON records
    # TODO: group events by (run_id, node_id, boot_id, sequence) for per-item lifecycle audit
    # TODO: for each lifecycle group, verify exactly one release event and one terminal event (ack/expire/drop)
    # TODO: count duplicate_releases, duplicate_terminals, terminal_without_release, unresolved_items
    # TODO: fit naive moving-average baseline on calibration data only
    # TODO: fit network-only model: features = [delivery_outcome, link_rssi, traffic_load]
    # TODO: fit cross-layer model from the frozen network, MAC, queue, and RTOS
    # feature allowlist
    # TODO: read manifest-defined calibration and held-out blocks; never invent a percentage split after seeing results
    # TODO: score all three models on identical held-out horizons: relative P95 error on deadline delivery ratio
    # TODO: compute primary uncertainty from run-level summaries or a whole-run cluster bootstrap
    # TODO: use within-run block bootstrap only for paired time-series uncertainty,
    # never as independent physical replication
    # TODO: compute prediction interval coverage: fraction of observations within predicted +/- 2 sigma
    # TODO: build a calibration-only support envelope and retain inside/outside status for every scored horizon
    # TODO: retain observation-integrity status; missing/stale/unreconciled horizons must not disappear silently
    # TODO: perform feature-group ablation only after the primary three-model comparison is frozen
    # TODO: generate gate characterization: state/reason vs time, trust fraction,
    # false trust, abstention/requalification latency, and P[2][2]
    # TODO: output the frozen primary metric table as CSV
    # TODO: exit nonzero if reconciliation fails (any lifecycle inconsistency)
    # TODO: use numpy for statistics, matplotlib for plots, scipy.stats for bootstrap

    print("ERROR: reproduction pipeline is a scaffold and produced no result.", file=sys.stderr)
    raise SystemExit(2)

if __name__ == "__main__":
    main()


## 5. Urutan Implementasi `reproduce.py`

Setiap langkah di bawah menyelesaikan TODO yang memang ada pada source. Urutannya sengaja fail-fast: structural evidence diperiksa sebelum fitting atau plotting.

1. **Parse input dan resolve boundary.** Path dinormalisasi, symlink/escape ditolak sesuai contract, directory harus ada, dan usage error memakai exit code yang dibekukan.
2. **Load `manifest.json` sebagai bytes dan JSON.** Decode error, duplicate key bila parser policy melarangnya, dan unexpected type menjadi failure yang jelas.
3. **Admission manifest.** `state` harus `ready`, `_todo` kosong, seluruh required field terisi, dan strict instance valid terhadap `schemas/experiment.schema.json`.
4. **Verify `manifest.sha256`.** Digest dihitung dari bytes manifest yang diarsipkan, bukan hasil reserialization.
5. **Load identity evidence.** Source revision, binary hashes, toolchain, board labels, roles, boot IDs, model/profile identity, dan key identity harus tersedia dari contract evidence Phase 3–5.
6. **Stream `events.ndjson`.** Satu object per nonblank line, line number dipertahankan dalam error, trailing garbage ditolak, dan file tidak diubah.
7. **Validate every record.** Required key, type, unit, bounded length, vocabulary, timestamp domain, serta payload encoding diperiksa sebelum grouping.
8. **Bind identity.** Record dengan `run_id`, node, boot, source, atau manifest identity yang tidak cocok tidak boleh masuk model.
9. **Lifecycle grouping.** Key yang sudah tertulis di source—`run_id`, `node_id`, `boot_id`, `sequence`—dipakai konsisten dengan protocol contract.
10. **Reconcile.** Tepat satu release dan satu terminal `ack`/`expire`/`drop` per item; duplicates, terminal-without-release, dan unresolved item dihitung serta ditampilkan.
11. **Load collection plan.** Calibration dan held-out berasal dari freeze record, bukan percentage split yang dipilih setelah plot dilihat.
12. **Construct horizons.** Semua model menerima horizon, target, exclusions, integrity status, dan support status yang sama.
13. **Fit calibration-only.** Naive moving average, network-only, dan frozen cross-layer model dilatih hanya pada allowed calibration data.
14. **Score held-out.** Relative P95 error atas deadline-delivery ratio memakai denominator guard dan definisi quantile frozen.
15. **Uncertainty.** Physical replication diringkas di run level atau whole-run cluster bootstrap; block bootstrap di dalam run hanya untuk paired time-series uncertainty.
16. **Integrity dan support.** Missing, stale, unreconciled, serta out-of-support horizons tetap berada di output dengan status, bukan dihapus.
17. **Gate characterization.** State/reason timeline, trust fraction, false trust, abstention/requalification latency, dan `P[2][2]` dihitung hanya bila evidence Phase 5 eligible.
18. **Atomic output dan exit.** Derived output ditulis ke directory terpisah, menyertakan input digests/configuration, dan tidak mengganti output existing tanpa explicit policy.

Ablation berjalan **setelah** primary three-model comparison frozen dan hanya bila core selesai. Jika tidak eligible, script mencatat `not run: eligibility reason`; ia tidak membuat hasil kosong seolah eksperimen dilaksanakan.


### Known-Evidence Fixtures sebelum Data Final

Reproduction membutuhkan fixture kecil yang berasal dari contract nyata, bukan angka final buatan. Fixture minimal mencakup:

1. satu lifecycle valid dengan satu release dan satu terminal;
2. duplicate release;
3. duplicate terminal;
4. terminal tanpa release;
5. release tanpa terminal;
6. malformed JSON pada line yang diketahui;
7. blank/trailing non-JSON record;
8. manifest digest mismatch;
9. `state: template` atau `_todo` tidak kosong;
10. record dari run/node/boot identity yang salah;
11. stale/missing horizon yang tetap muncul sebagai non-evaluable;
12. frozen tiny calibration/held-out example dengan expected metric dihitung tangan.

Expected outcome fixture berupa verdict, counter, atau exit code yang dideklarasikan—bukan hasil dari script itu sendiri. Final raw evidence baru dianalisis setelah fixture suite lulus.


## 6. Recorder dan Evidence Contract

Recorder adalah frozen dependency Phase 3, tetapi Week 6 bergantung penuh padanya. Header saat ini hanya menerima manifest dan raw records; implementation TODO menjanjikan `manifest.sha256`, `versions` placeholder, `events.ndjson`, serta terminal `run-status.json`. Sebelum final campaign, data yang dijanjikan oleh reproduction harus benar-benar dapat dihasilkan atau dikumpulkan melalui owner yang jelas.

Checklist contract:

1. `cldt_run_recorder_open` gagal pada template, run ID tidak valid, manifest kosong, unsafe path, dan existing directory.
2. Directory dibuat atomically; manifest disalin byte-for-byte dan digest dihitung dari salinan tersebut.
3. Identity yang tidak masuk API recorder mempunyai sumber dan pengikatan yang eksplisit; tidak boleh bergantung pada catatan ingatan operator.
4. `cldt_run_recorder_append` menulis satu newline-terminated JSON record, escaping/encoding benar, durability policy terdokumentasi, dan counter naik hanya setelah write berhasil.
5. `cldt_run_recorder_finalize` menerima fixed status/reason vocabulary, atomic terminal status, flush/close, dan second-finalize rejection.
6. Interrupted process tidak “dibersihkan”; recovery procedure mengklasifikasikan directory secara append-only tanpa menulis ulang raw event.
7. Exact `events.ndjson` object contract dipakai sama oleh recorder dan reproduction.

Jika recorder contract belum mampu menyimpan identity lengkap, Week 6 tidak boleh menambalnya dengan nama file manual yang tidak punya owner. Kekurangan tersebut adalah evidence defect yang harus diselesaikan sebelum final runs.


### `host/run_recorder.h`

Cell berikut adalah salinan persis dari `host/run_recorder.h` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/run_recorder.h
#ifndef CLDT_HOST_RUN_RECORDER_H
#define CLDT_HOST_RUN_RECORDER_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>
#include <stdio.h>

#include "cldt/cldt_status.h"
#include "experiment_config.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef struct {
    /* event_stream is append-only. finalized prevents a second terminal status. */
    FILE *event_stream;
    char run_directory[260];
    uint64_t records_written;
    bool finalized;
} cldt_run_recorder_t;

/*
 * Creates a new run directory and fails if it already exists. The recorder owns
 * only files it creates for this run; it may never delete or rewrite a prior run.
 */
cldt_status_t cldt_run_recorder_open(
    cldt_run_recorder_t *recorder,
    const char *results_root,
    const cldt_experiment_config_t *config,
    const char *original_manifest,
    size_t original_manifest_bytes);

/* Appends one raw received record; parsing and model updates happen elsewhere. */
cldt_status_t cldt_run_recorder_append(
    cldt_run_recorder_t *recorder,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint64_t received_host_us);

/* Writes one immutable terminal status after counters and evidence are collected. */
cldt_status_t cldt_run_recorder_finalize(
    cldt_run_recorder_t *recorder,
    const char *status,
    const char *reason);

#ifdef __cplusplus
}
#endif

#endif


### `host/run_recorder.c`

Cell berikut adalah salinan persis dari `host/run_recorder.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/run_recorder.c
#include "run_recorder.h"

cldt_status_t cldt_run_recorder_open(
    cldt_run_recorder_t *recorder,
    const char *results_root,
    const cldt_experiment_config_t *config,
    const char *original_manifest,
    size_t original_manifest_bytes)
{
    (void)recorder;
    (void)results_root;
    (void)config;
    (void)original_manifest;
    (void)original_manifest_bytes;

    /*
     * IMPLEMENTATION TODO:
     * 1. Reject a template config, zero/unreserved run ID, unsafe path component,
     *    empty manifest, or pre-existing target directory. Derive the directory
     *    name from the reserved run identifier plus timestamp, never from
     *    unchecked user input. Record the coordinator boot ID, ledger identity,
     *    and, for actuation, the non-secret command-key identity so nonce
     *    provenance is auditable.
     * 2. Create the directory atomically, copy the original manifest byte-for-
     *    byte, write its SHA-256 and a versions placeholder that will resolve
     *    source/binary identities plus the selected control_profile, then fsync
     *    metadata before accepting observations.
     * 3. Open events.ndjson in append-only mode and leave finalized false. If any
     *    step fails, roll back only the newly created empty directory.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_run_recorder_append(
    cldt_run_recorder_t *recorder,
    const char *topic,
    const uint8_t *payload,
    size_t payload_bytes,
    uint64_t received_host_us)
{
    (void)recorder;
    (void)topic;
    (void)payload;
    (void)payload_bytes;
    (void)received_host_us;

    /*
     * IMPLEMENTATION TODO: require an open non-finalized recorder, validate a
     * bounded topic and payload length, encode binary payload safely (for example
     * base64), escape all JSON strings, and append exactly one newline-terminated
     * record containing host receive time. Flush according to a documented
     * durability policy and increment records_written only after a successful
     * write. Do not parse, reorder, or discard raw evidence in this layer.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}

cldt_status_t cldt_run_recorder_finalize(
    cldt_run_recorder_t *recorder,
    const char *status,
    const char *reason)
{
    (void)recorder;
    (void)status;
    (void)reason;

    /*
     * IMPLEMENTATION TODO: accept only a fixed status vocabulary and predefined
     * exclusion reasons, write one small run-status.json atomically, flush and
     * close the event stream, then set finalized true. A second finalize call
     * must fail without changing files. Never reopen or rewrite events.ndjson
     * during finalization, even when the run is invalid or interrupted.
     */
    return CLDT_ERR_NOT_IMPLEMENTED;
}


## 7. Build dan Deterministic C Verification

Build closure memisahkan tiga fakta: configure berhasil, compile/link berhasil, dan assertions benar-benar berjalan. `tests/CMakeLists.txt` mendaftarkan `SKIP_RETURN_CODE 77`; karena itu exit 0 dari `ctest` belum cukup bila tujuh test masih skipped.

Urutan verifikasi:

1. Configure clean build dari source revision frozen.
2. Compile dengan warning policy yang sama dengan final environment.
3. Jalankan `ctest --test-dir build --output-on-failure`.
4. Baca summary dan individual exit status: target akhir adalah tujuh test aktif, nol failed, dan **nol skipped**.
5. Jalankan negative vectors yang sudah menjadi contract Phase 1–5, bukan hanya happy path.
6. Arsipkan command, tool versions, source revision, timestamp, dan complete log sebagai build evidence.
7. Ulangi dari clean clone pada mesin/environment yang dipakai untuk handoff.

Build directory tetap disposable dan bukan raw scientific evidence.


### `CMakeLists.txt`

Cell berikut adalah salinan persis dari `CMakeLists.txt` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/root_CMakeLists.txt
cmake_minimum_required(VERSION 3.20)

project(cldt_host LANGUAGES C)

option(CLDT_BUILD_TESTS "Build the host-side skeletal tests" OFF)

set(CMAKE_C_STANDARD 11)
set(CMAKE_C_STANDARD_REQUIRED ON)
set(CMAKE_C_EXTENSIONS OFF)

add_subdirectory(common)
add_subdirectory(host)

if(CLDT_BUILD_TESTS)
    enable_testing()
    add_subdirectory(tests)
endif()


### `common/CMakeLists.txt`

Cell berikut adalah salinan persis dari `common/CMakeLists.txt` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/common_CMakeLists.txt
set(CLDT_COMMON_SOURCES
    "src/cldt_protocol.c"
    "src/cldt_clock_sync.c"
    "src/cldt_control_profile.c"
    "src/cldt_metrics.c"
    "src/cldt_event_trace.c"
    "src/cldt_auth.c"
    "src/cldt_crc32c.c"
)

if(COMMAND idf_component_register)
    idf_component_register(
        SRCS ${CLDT_COMMON_SOURCES}
        INCLUDE_DIRS "include"
    )
else()
    add_library(cldt_common STATIC ${CLDT_COMMON_SOURCES})
    target_include_directories(cldt_common
        PUBLIC
            ${CMAKE_CURRENT_SOURCE_DIR}/include
    )

    if(MSVC)
        target_compile_options(cldt_common PRIVATE /W4)
    else()
        target_compile_options(cldt_common PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
    endif()
endif()


### `host/CMakeLists.txt`

Cell berikut adalah salinan persis dari `host/CMakeLists.txt` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/host_CMakeLists.txt
add_executable(cldt_host
    main.c
    coordinator.c
    experiment_config.c
    twin_model.c
    estimator.c
    fidelity_gate.c
    policy.c
    broker_io.c
    run_recorder.c
    kalman.c
)

target_include_directories(cldt_host PRIVATE ${CMAKE_CURRENT_SOURCE_DIR})
target_link_libraries(cldt_host PRIVATE cldt_common)

if(MSVC)
    target_compile_options(cldt_host PRIVATE /W4)
else()
    target_compile_options(cldt_host PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
endif()


### `tests/CMakeLists.txt`

Cell berikut adalah salinan persis dari `tests/CMakeLists.txt` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/tests_CMakeLists.txt
function(cldt_add_skeletal_test name source)
    add_executable(${name} ${source})
    target_link_libraries(${name} PRIVATE cldt_common)
    add_test(NAME ${name} COMMAND ${name})
    set_tests_properties(${name} PROPERTIES SKIP_RETURN_CODE 77)
endfunction()

cldt_add_skeletal_test(test_protocol test_protocol.c)
cldt_add_skeletal_test(test_crc32c test_crc32c.c)
cldt_add_skeletal_test(test_auth test_auth.c)
cldt_add_skeletal_test(test_clock_sync test_clock_sync.c)
cldt_add_skeletal_test(test_metrics test_metrics.c)
cldt_add_skeletal_test(test_event_trace test_event_trace.c)
cldt_add_skeletal_test(test_control_profile test_control_profile.c)


### Tujuh Existing Test Owners

Tidak ada test file baru yang diberi nama oleh notebook. Tujuh owner yang benar-benar ada harus menutup assertions dari Phase 1–5:

| File | Contract yang dibuktikan | Failure penting |
|---|---|---|
| `tests/test_protocol.c` | legal frame encode/decode dan size/version/type boundary | malformed length, illegal type/version, buffer boundary |
| `tests/test_crc32c.c` | fixed CRC-32C vectors dan corruption detection | wrong polynomial/init/finalization, altered byte |
| `tests/test_auth.c` | authenticated command vector dan rejection | tag/key/nonce/AAD corruption, replay-related identity |
| `tests/test_clock_sync.c` | wrap-aware timestamp/order conversion | wrap boundary, invalid or stale mapping |
| `tests/test_metrics.c` | counter accounting dan deadline ratio | zero denominator, overflow, inconsistent totals |
| `tests/test_event_trace.c` | event/lifecycle trace vocabulary dan bounds | illegal event, invalid sequence/payload |
| `tests/test_control_profile.c` | resolved profile limits dan fail-closed admission | illegal TTL/rate/freshness/hysteresis combination |

Setiap cell berikut mempertahankan TODO/skip persis seperti repo. Week 6 tidak mengganti unit tests dengan log hardware.


### `tests/test_protocol.c`

Cell berikut adalah salinan persis dari `tests/test_protocol.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_protocol.c
#include <stdio.h>

#include "cldt/cldt_protocol.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Begin with fixed hexadecimal byte vectors for the smallest and largest
     *    legal frames. Assert every CLDT_WIRE_*_OFFSET, zero reserved bytes,
     *    network byte order, exact size, CRC-32C, and authentication
     *    tag—not only encode/decode round trips, which can hide matching mistakes
     *    on both sides. Mutating either reserved byte must fail decoding.
     * 2. Add rejection cases one mutation at a time: wrong magic, unsupported
     *    version, header/payload length mismatch, truncation at every boundary,
     *    trailing bytes, CRC mutation, authentication mutation, and oversize data.
     * 3. Build one fixed CLDT_POLICY_WIRE_BYTES vector and assert every policy
     *    array/field offset plus equality between payload and metadata epochs.
     *    Test wrong run ID, duplicate epoch, older epoch, stale issue time, zero
     *    TTL, expired TTL, and boundary uncertainty. Verify the exact status,
     *    including CLDT_ERR_STALE and CLDT_ERR_WRONG_RUN, and confirm decoder
     *    output is not partially published.
     *    Gateway integration also rejects a nonzero command-authority node ID or
     *    wrong commissioned authority boot ID without re-encoding the datagram.
     * 4. Keep test vectors in ordinary source data with a short derivation note.
     *    Do not connect Thread or MQTT until these host-only checks are green.
     */
    fprintf(stderr, "SKIP: protocol tests have not been implemented.\n");
    return 77;
}


### `tests/test_crc32c.c`

Cell berikut adalah salinan persis dari `tests/test_crc32c.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_crc32c.c
#include <stdio.h>

#include "cldt/cldt_crc32c.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Freeze standard CRC-32C known-answer vectors, including empty input and
     *    "123456789", with the exact seed and final-XOR convention.
     * 2. Test incremental versus single-buffer updates, zero length, unaligned
     *    input, binary zero bytes, and the canonical header-plus-payload wire
     *    integrity sequence.
     * 3. Run identical vectors on the host, ESP32-S3, and ESP32-C6. A platform
     *    helper with the IEEE polynomial must fail the Castagnoli vector.
     * 4. Do not enable frame acceptance until this test is no longer skipped.
     */
    fprintf(stderr, "SKIP: CRC-32C tests have not been implemented.\n");
    return 77;
}


### `tests/test_auth.c`

Cell berikut adalah salinan persis dari `tests/test_auth.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_auth.c
#include <stdio.h>

#include "cldt/cldt_auth.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Use RFC 8439 known-answer material plus one project-specific fixed
     *    command vector with authority node ID 0, a fixed coordinator boot ID,
     *    the normative AAD byte order, and zero plaintext.
     * 2. Mutate every authenticated region, tag byte, run ID, epoch, and nonce
     *    byte independently and assert exact authentication failure status.
     *    A validly tagged but non-commissioned coordinator boot ID must produce
     *    CLDT_ERR_WRONG_AUTHORITY at the state-validation boundary.
     * 3. Verify one immutable command per epoch, strict epoch advance, and that
     *    retransmission reuses identical authenticated bytes rather than
     *    generating a different command under the same nonce.
     * 4. In the endpoint integration suite, verify persist-before-apply, reboot
     *    from a valid highest-epoch record, and safe fallback for missing,
     *    corrupt, or unwritable replay state. Boot identity must not substitute
     *    for that state.
     * 5. Run the same vector through the host and mbedTLS-backed targets before
     *    provisioning a command key or enabling remote actuation.
     */
    fprintf(stderr, "SKIP: authentication tests have not been implemented.\n");
    return 77;
}


### `tests/test_clock_sync.c`

Cell berikut adalah salinan persis dari `tests/test_clock_sync.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_clock_sync.c
#include <stdio.h>

#include "cldt/cldt_clock_sync.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Use synthetic four-timestamp exchanges with a known offset and drift.
     *    Assert that the estimate remains invalid until the documented minimum
     *    sample rule is met, then maps local time within its reported uncertainty.
     * 2. Add asymmetric-delay and high-round-trip samples. Verify that the chosen
     *    filter either rejects them or expands uncertainty; it must not return a
     *    deceptively precise one-way time.
     * 3. Test timestamp ordering faults, arithmetic near integer boundaries,
     *    boot/reset behavior, drift over a long interval, and output pointers
     *    remaining unchanged on error.
     * 4. The success criterion is honest uncertainty propagation, not merely a
     *    small offset on an ideal synthetic clock.
     */
    fprintf(stderr, "SKIP: clock-sync tests have not been implemented.\n");
    return 77;
}


### `tests/test_metrics.c`

Cell berikut adalah salinan persis dari `tests/test_metrics.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_metrics.c
#include <stdio.h>

#include "cldt/cldt_metrics.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Construct trace records carrying traffic class, run ID, node ID, boot
     *    ID, sequence, release/deadline/current timestamps, and terminal event.
     *    Assert the expected counter and checked timing delta after every record;
     *    test every kind through CLDT_EVENT_COUNT, including explicit coalescing
     *    and pool exhaustion plus policy events that must not inflate delivery.
     * 2. Build conservation cases for acknowledged, expired, coalesced, rejected,
     *    dropped, duplicated, and genuinely unresolved work. Verify the report
     *    distinguishes an incomplete run from a mathematically inconsistent one.
     * 3. Sort raw records by the documented full identity and test the item audit
     *    with acknowledgement before release, two terminal outcomes for one item,
     *    a terminal without release, unresolved work, invalid class/kind, and
     *    out-of-order input. Include the counterbalanced case where item A has two
     *    terminals and item B has none: aggregate totals may balance, but the item
     *    audit must fail.
     * 4. Add counter saturation and reset-at-run-boundary cases. Do not calculate
     *    PDR, deadline ratio, or energy efficiency until both aggregate
     *    reconciliation and per-item audit succeed.
     */
    fprintf(stderr, "SKIP: metric-accounting tests have not been implemented.\n");
    return 77;
}


### `tests/test_event_trace.c`

Cell berikut adalah salinan persis dari `tests/test_event_trace.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_event_trace.c
#include <stdio.h>

#include "cldt/cldt_event_trace.h"
#include "cldt/cldt_metrics.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Build deterministic traces for release, admission, dequeue, ACK,
     *    expiry, rejection, coalescing, drop, restart, and fallback events.
     * 2. Replay one injected fault at a time: truncation, corruption,
     *    duplication, reordering, missing terminal, stale observation, wrong
     *    run, boot-ID change, and durable replay-state loss.
     * 3. Assert item-level audit and aggregate reconciliation independently;
     *    layer-specific MAC attempt/ACK diagnostics are not forced into false
     *    equality with application messages.
     * 4. Assert missing or unreconciled evidence produces an explicit invalid
     *    observation input and can never become a favorable gate sample.
     */
    fprintf(stderr, "SKIP: event-trace and deterministic replay tests have not been implemented.\n");
    return 77;
}


### `tests/test_control_profile.c`

Cell berikut adalah salinan persis dari `tests/test_control_profile.c` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/test_control_profile.c
#include <stdio.h>

#include "cldt/cldt_control_profile.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Start with one completely specified in-memory profile whose IDs are
     *    bounded, digest is nonzero, and host/edge limits are all finite.
     * 2. Test one invalid condition at a time: null pointer, empty ID, missing
     *    NUL terminator, invalid/non-v1 actuation model, all-zero digest, zero
     *    freshness window, zero TTL, zero rate ceiling, zero critical period,
     *    and zero bulk burst ceiling.
     * 3. Assert exact status codes and assert the validator has not changed the
     *    input bytes. The test must not open a profile file or contact a device.
     * 4. Add a host-level test later for a manifest/profile-ID mismatch; that
     *    belongs above this portable common-library test.
     */
    fprintf(stderr, "SKIP: control profile tests have not been implemented.\n");
    return 77;
}


## 8. CI Closure dan Batas Klaim

Workflow repository memeriksa host build/test surface, schema/template shape, exact ID pair, dan scaffold guard. Ia belum membuktikan:

- test tidak skipped;
- strict manifest sudah `ready` dan `_todo` kosong;
- cross-field eligibility sesuai experiment;
- firmware build/flash;
- physical Thread attachment;
- recorder/reproduction end to end;
- clean-clone scientific result;
- GitHub runner benar-benar memulai job.

Karena itu CI evidence mencatat run URL/ID, head SHA, event, start/completion, job conclusion, dan annotation. Kegagalan runner/billing/permission dibedakan dari code failure; keduanya tidak boleh disebut “CI pass.” Local evidence tetap diperlukan, tetapi tidak dipakai untuk memalsukan remote status.


### `.github/workflows/scaffold-validation.yml`

Cell berikut adalah salinan persis dari `.github/workflows/scaffold-validation.yml` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/scaffold-validation.yml
name: Scaffold Validation

on:
  push:
    branches:
      - main
  pull_request:

permissions:
  contents: read

jobs:
  host-and-manifests:
    name: Host Build and Manifest Contracts
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Configure host scaffold
        run: cmake -S . -B build -DCLDT_BUILD_TESTS=ON

      - name: Build host scaffold
        run: cmake --build build --parallel

      - name: Run test harness
        run: ctest --test-dir build --output-on-failure

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install JSON Schema validator
        run: python -m pip install --disable-pip-version-check "jsonschema==4.26.0"

      - name: Validate experiment manifests
        shell: bash
        run: |
          python - <<'PY'
          import json
          from pathlib import Path

          from jsonschema import Draft202012Validator

          schema_path = Path("schemas/experiment.schema.json")
          schema = json.loads(schema_path.read_text(encoding="utf-8"))
          Draft202012Validator.check_schema(schema)
          validator = Draft202012Validator(schema)

          strict_files = sorted(Path("experiments").glob("*.json"))
          if not strict_files:
              raise SystemExit("No strict experiment manifests were found.")

          strict_ids = {}
          for path in strict_files:
              document = json.loads(path.read_text(encoding="utf-8"))
              errors = sorted(validator.iter_errors(document), key=lambda error: list(error.path))
              if errors:
                  for error in errors:
                      location = "/" + "/".join(str(part) for part in error.path)
                      print(f"{path}:{location}: {error.message}")
                  raise SystemExit(f"Schema validation failed for {path}.")
              strict_ids[path.stem] = document["experiment_id"]

          authoring_files = sorted(Path("experiments/authoring").glob("*.jsonc"))
          if len(authoring_files) != len(strict_files):
              raise SystemExit("Strict JSON and JSONC authoring manifest counts differ.")

          for path in authoring_files:
              uncommented = "\n".join(
                  line for line in path.read_text(encoding="utf-8").splitlines()
                  if not line.lstrip().startswith("//")
              )
              document = json.loads(uncommented)
              if document["experiment_id"] != strict_ids.get(path.stem):
                  raise SystemExit(f"Experiment ID mismatch for {path}.")

          print(f"Validated {len(strict_files)} strict manifests and {len(authoring_files)} authoring copies.")
          PY


## 9. Schema Admission dan Strict Manifest Pair

Schema mengizinkan dua state: template boleh memiliki null dan actionable `_todo`; ready harus memiliki `_todo: []` dan seluruh contract field terisi. Schema validation adalah syarat perlu, bukan cukup. Week-6 admission juga memeriksa:

1. strict JSON, bukan JSONC, menjadi manifest run;
2. authoring/strict pair mempunyai `schema_version` dan `experiment_id` yang sama;
3. digest strict JSON dihitung sebelum run dan salinannya masuk run directory;
4. `state` baru berubah ke `ready` setelah seluruh value berasal dari frozen decision/evidence;
5. treatment legal untuk experiment—misalnya load-step tetap shadow-only;
6. setup identik dengan evidence block yang dirujuk;
7. `repetitions` dan `seed` adalah plan frozen, bukan hasil script;
8. `required_artifacts` hanya memuat artefak yang owner-nya benar-benar menghasilkan;
9. setiap perubahan setelah digest menghasilkan manifest identity baru.

Nilai null tidak diisi dengan contoh notebook. Isinya berasal dari topology record, pilot, source/binary hash, experiment plan, dan acceptance yang telah dibekukan.


### `schemas/experiment.schema.json`

Cell berikut adalah salinan persis dari `schemas/experiment.schema.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/experiment.schema.json
{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "https://github.com/Reinathanajah/CLDT-Thread/blob/main/schemas/experiment.schema.json",
  "title": "CLDT Experiment Manifest",
  "description": "A deliberately small, two-stage contract. A template may retain null decisions and actionable _todo entries; a ready manifest may not.",
  "type": "object",
  "additionalProperties": false,
  "required": ["schema_version", "state", "experiment_id", "title", "purpose", "_todo"],
  "properties": {
    "schema_version": { "const": "2.0" },
    "state": { "enum": ["template", "ready"] },
    "experiment_id": { "$ref": "#/$defs/identifier" },
    "title": { "type": "string", "minLength": 8, "maxLength": 120 },
    "purpose": { "$ref": "#/$defs/purpose" },
    "_todo": {
      "type": "array",
      "uniqueItems": true,
      "items": { "$ref": "#/$defs/todo" }
    },
    "setup": { "$ref": "#/$defs/template_setup" },
    "execution": { "$ref": "#/$defs/template_execution" },
    "traffic": { "$ref": "#/$defs/template_traffic" },
    "scenario": { "$ref": "#/$defs/template_scenario" },
    "treatment": { "$ref": "#/$defs/template_treatment" },
    "acceptance": { "$ref": "#/$defs/template_acceptance" },
    "evidence": { "$ref": "#/$defs/template_evidence" }
  },
  "allOf": [
    {
      "if": { "properties": { "state": { "const": "template" } } },
      "then": { "properties": { "_todo": { "minItems": 1 } } }
    },
    {
      "if": { "properties": { "state": { "const": "ready" } } },
      "then": {
        "required": ["setup", "execution", "traffic", "scenario", "treatment", "acceptance", "evidence"],
        "properties": {
          "_todo": { "maxItems": 0 },
          "setup": { "$ref": "#/$defs/ready_setup" },
          "execution": { "$ref": "#/$defs/ready_execution" },
          "traffic": { "$ref": "#/$defs/ready_traffic" },
          "scenario": { "$ref": "#/$defs/ready_scenario" },
          "treatment": { "$ref": "#/$defs/ready_treatment" },
          "acceptance": { "$ref": "#/$defs/ready_acceptance" },
          "evidence": { "$ref": "#/$defs/ready_evidence" }
        }
      }
    }
  ],
  "$defs": {
    "identifier": {
      "type": "string",
      "minLength": 3,
      "maxLength": 64,
      "pattern": "^[a-z0-9][a-z0-9._-]*$"
    },
    "non_empty_text": {
      "type": "string",
      "minLength": 3,
      "maxLength": 512
    },
    "purpose": {
      "type": "object",
      "additionalProperties": false,
      "required": ["question", "comparison", "primary_metric"],
      "properties": {
        "question": { "$ref": "#/$defs/non_empty_text" },
        "comparison": { "$ref": "#/$defs/non_empty_text" },
        "primary_metric": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "todo": {
      "type": "object",
      "additionalProperties": false,
      "required": ["path", "action", "method", "done_when"],
      "properties": {
        "path": { "type": "string", "pattern": "^/" },
        "action": { "$ref": "#/$defs/non_empty_text" },
        "method": { "$ref": "#/$defs/non_empty_text" },
        "done_when": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "node": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "board", "role"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "board": { "enum": ["esp32-s3", "esp32-c6"] },
        "role": { "enum": ["gateway", "radio_coprocessor", "router_endpoint", "low_power_endpoint"] }
      }
    },
    "template_setup": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "nodes": { "type": ["array", "null"], "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/node" } },
        "thread_channel": { "type": ["integer", "null"], "minimum": 11, "maximum": 26 },
        "placement": { "type": ["string", "null"], "maxLength": 160 },
        "firmware_reference": { "type": ["string", "null"], "maxLength": 160 }
      }
    },
    "ready_setup": {
      "type": "object",
      "additionalProperties": false,
      "required": ["nodes", "thread_channel", "placement", "firmware_reference"],
      "properties": {
        "nodes": { "type": "array", "minItems": 4, "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/node" } },
        "thread_channel": { "type": "integer", "minimum": 11, "maximum": 26 },
        "placement": { "$ref": "#/$defs/non_empty_text" },
        "firmware_reference": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_execution": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "warmup_s": { "type": ["integer", "null"], "minimum": 5, "maximum": 300 },
        "measurement_s": { "type": ["integer", "null"], "minimum": 30, "maximum": 3600 },
        "cooldown_s": { "type": ["integer", "null"], "minimum": 5, "maximum": 300 },
        "repetitions": { "type": ["integer", "null"], "minimum": 1, "maximum": 30 },
        "seed": { "type": ["integer", "null"], "minimum": 0, "maximum": 4294967295 }
      }
    },
    "ready_execution": {
      "type": "object",
      "additionalProperties": false,
      "required": ["warmup_s", "measurement_s", "cooldown_s", "repetitions", "seed"],
      "properties": {
        "warmup_s": { "type": "integer", "minimum": 5, "maximum": 300 },
        "measurement_s": { "type": "integer", "minimum": 30, "maximum": 3600 },
        "cooldown_s": { "type": "integer", "minimum": 5, "maximum": 300 },
        "repetitions": { "type": "integer", "minimum": 1, "maximum": 30 },
        "seed": { "type": "integer", "minimum": 0, "maximum": 4294967295 }
      }
    },
    "stream": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "source", "class", "period_ms", "payload_bytes", "deadline_ms", "burst_packets"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "source": { "$ref": "#/$defs/identifier" },
        "class": { "enum": ["control", "critical", "telemetry", "bulk"] },
        "period_ms": { "type": ["integer", "null"], "minimum": 10, "maximum": 3600000 },
        "payload_bytes": { "type": ["integer", "null"], "minimum": 1, "maximum": 256 },
        "deadline_ms": { "type": ["integer", "null"], "minimum": 10, "maximum": 3600000 },
        "burst_packets": { "type": ["integer", "null"], "minimum": 1, "maximum": 100 }
      }
    },
    "ready_stream": {
      "type": "object",
      "additionalProperties": false,
      "required": ["id", "source", "class", "period_ms", "payload_bytes", "deadline_ms", "burst_packets"],
      "properties": {
        "id": { "$ref": "#/$defs/identifier" },
        "source": { "$ref": "#/$defs/identifier" },
        "class": { "enum": ["control", "critical", "telemetry", "bulk"] },
        "period_ms": { "type": "integer", "minimum": 10, "maximum": 3600000 },
        "payload_bytes": { "type": "integer", "minimum": 1, "maximum": 256 },
        "deadline_ms": { "type": "integer", "minimum": 10, "maximum": 3600000 },
        "burst_packets": { "type": "integer", "minimum": 1, "maximum": 100 }
      }
    },
    "template_traffic": {
      "type": "object",
      "additionalProperties": false,
      "properties": { "streams": { "type": ["array", "null"], "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/stream" } } }
    },
    "ready_traffic": {
      "type": "object",
      "additionalProperties": false,
      "required": ["streams"],
      "properties": { "streams": { "type": "array", "minItems": 1, "maxItems": 4, "uniqueItems": true, "items": { "$ref": "#/$defs/ready_stream" } } }
    },
    "template_scenario": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "event": { "enum": ["none", "load_step", "observation_pause", "endpoint_restart", "topology_shift", null] },
        "at_s": { "type": ["integer", "null"], "minimum": 0, "maximum": 3600 },
        "duration_s": { "type": ["integer", "null"], "minimum": 0, "maximum": 3600 },
        "target": { "type": ["string", "null"], "maxLength": 120 }
      }
    },
    "ready_scenario": {
      "type": "object",
      "additionalProperties": false,
      "required": ["event", "at_s", "duration_s", "target"],
      "properties": {
        "event": { "enum": ["none", "load_step", "observation_pause", "endpoint_restart", "topology_shift"] },
        "at_s": { "type": "integer", "minimum": 0, "maximum": 3600 },
        "duration_s": { "type": "integer", "minimum": 0, "maximum": 3600 },
        "target": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_treatment": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "mode": { "enum": ["baseline", "prediction", "gated_control", "safety", "smp", "power", null] },
        "candidate_action": { "enum": ["none", "bulk_rate_reduce", "phase_stagger", "power_profile", null] },
        "control_profile": { "type": ["string", "null"], "maxLength": 160 },
        "host_model": { "type": ["boolean", "null"] },
        "remote_actuation": { "type": ["boolean", "null"] }
      }
    },
    "ready_treatment": {
      "type": "object",
      "additionalProperties": false,
      "required": ["mode", "candidate_action", "control_profile", "host_model", "remote_actuation"],
      "properties": {
        "mode": { "enum": ["baseline", "prediction", "gated_control", "safety", "smp", "power"] },
        "candidate_action": { "enum": ["none", "bulk_rate_reduce", "phase_stagger", "power_profile"] },
        "control_profile": { "$ref": "#/$defs/non_empty_text" },
        "host_model": { "type": "boolean" },
        "remote_actuation": { "type": "boolean" }
      }
    },
    "template_acceptance": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "counter_reconciliation": { "type": ["boolean", "null"] },
        "minimum_critical_on_time_pdr": { "type": ["number", "null"], "minimum": 0, "maximum": 1 },
        "negative_case": { "type": ["string", "null"], "maxLength": 120 }
      }
    },
    "ready_acceptance": {
      "type": "object",
      "additionalProperties": false,
      "required": ["counter_reconciliation", "minimum_critical_on_time_pdr", "negative_case"],
      "properties": {
        "counter_reconciliation": { "const": true },
        "minimum_critical_on_time_pdr": { "type": "number", "minimum": 0, "maximum": 1 },
        "negative_case": { "$ref": "#/$defs/non_empty_text" }
      }
    },
    "template_evidence": {
      "type": "object",
      "additionalProperties": false,
      "properties": {
        "required_artifacts": { "type": ["array", "null"], "maxItems": 8, "uniqueItems": true, "items": { "$ref": "#/$defs/non_empty_text" } },
        "operator_notes_required": { "type": ["boolean", "null"] },
        "topology_photo_required": { "type": ["boolean", "null"] }
      }
    },
    "ready_evidence": {
      "type": "object",
      "additionalProperties": false,
      "required": ["required_artifacts", "operator_notes_required", "topology_photo_required"],
      "properties": {
        "required_artifacts": { "type": "array", "minItems": 4, "maxItems": 8, "uniqueItems": true, "items": { "$ref": "#/$defs/non_empty_text" } },
        "operator_notes_required": { "const": true },
        "topology_photo_required": { "type": "boolean" }
      }
    }
  }
}


## 10. Archived Local-RTOS Prerequisite

`local-rtos-accounting` adalah prerequisite Phase 2, bukan otomatis diulang pada Week 6. Pair ini dibuka hanya untuk memeriksa bahwa final evidence chain merujuk block local accounting yang benar.

Urutan penyelesaian TODO manifest:

1. `/setup` diisi dari cold-boot local test tanpa radio: board identity, firmware reference, dan physical setup.
2. `/execution` diisi dari pilot lokal yang menetapkan phases, streams, seed, dan repetition plan.
3. `/acceptance` mengikat released/ack/expire/drop reconciliation tanpa membutuhkan network claim.
4. Strict partner menjadi `ready` hanya jika block ini memang sudah selesai dan diarsipkan.
5. Jika source/binary yang memengaruhi accounting berubah, prerequisite dijalankan ulang dalam identity block baru.

Week 6 tidak mengubah hasil lokal supaya cocok dengan network result.


### `experiments/authoring/local-rtos-baseline.jsonc`

Cell berikut adalah salinan persis dari `experiments/authoring/local-rtos-baseline.jsonc` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/local-rtos-baseline.jsonc
{
  // This is the authoring guide for ../local-rtos-baseline.json.
  // Thread and Wi-Fi must remain inactive for this local-accounting experiment.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "local-rtos-accounting",
  "title": "Local RTOS Accounting Baseline",
  "purpose": {
    "question": "Can one endpoint account for every released item before Thread networking is introduced?",
    "comparison": "Critical and bulk queue behavior on a local endpoint with networking disabled.",
    "primary_metric": "Zero unreconciled items across release, queue, expiry, and terminal accounting."
  },
  "setup": {
    // Use the actual endpoint label; retain later topology roles only as context, not as a dependency.
    "nodes": null,
    // Record the intended later Thread channel only if it is known; radio attachment remains disabled here.
    "thread_channel": null,
    // State bench position, power source, and that no Thread network is active.
    "placement": null,
    // Include endpoint source revision, sdkconfig hash, binary hash, and local-test build option.
    "firmware_reference": null
  },
  "execution": {
    // Warm-up allows timers and trace buffers to reach steady state before measurement.
    "warmup_s": null,
    // Include enough releases to exercise periodic critical work and bounded bulk burst behavior.
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    // The seed must reproduce workload release and any deterministic burst pattern.
    "seed": null
  },
  "traffic": {
    // Define one critical and one bulk stream with explicit period, deadline, payload, and burst.
    "streams": null
  },
  "scenario": {
    // Use event none unless one declared local queue disturbance is introduced.
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Local accounting is baseline behavior: no model and no remote policy.
    "mode": null,
    "candidate_action": null,
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    // Set true only after the implementation can reconcile each logical work item.
    "counter_reconciliation": null,
    // Choose a local service floor only if clock/deadline accounting is ready to support it.
    "minimum_critical_on_time_pdr": null,
    // Require an explicit invalidation rule such as missing terminal trace records.
    "negative_case": null
  },
  "evidence": {
    // Require raw local trace, counters, binary identity, manifest, and operator notes.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Create a reproducible local endpoint build.",
      "method": "Archive board label, firmware identity, power path, and proof that Thread attachment is disabled.",
      "done_when": "Cold boot reaches local test mode without radio attachment."
    },
    {
      "path": "/execution",
      "action": "Select timing from traceable pilots.",
      "method": "Exercise critical-only, bulk-only, and combined paths; capture release, admission, expiry, and terminal events.",
      "done_when": "Every intended path appears and no released item is unexplained."
    },
    {
      "path": "/acceptance",
      "action": "Turn the result into a real accounting gate.",
      "method": "Require reconciliation and retain evidence that lets another reader recompute terminal outcomes offline.",
      "done_when": "No network performance claim is needed to validate the local result."
    }
  ]
}


### `experiments/local-rtos-baseline.json`

Cell berikut adalah salinan persis dari `experiments/local-rtos-baseline.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/local-rtos-baseline.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "local-rtos-accounting",
  "title": "Local RTOS Accounting Baseline",
  "purpose": {
    "question": "Can one endpoint account for every released item before Thread networking is introduced?",
    "comparison": "Critical and bulk queue behavior on a local endpoint with networking disabled.",
    "primary_metric": "Zero unreconciled items across release, queue, expiry, and terminal accounting."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Define the one endpoint that will run the local queue test, while explicitly keeping Thread and Wi-Fi inactive.",
      "method": "Record the actual C6 board label, ESP-IDF revision, sdkconfig hash, binary hash, power source, and bench position. Set the four physical roles only if the later Thread topology is already assembled; this local run must not depend on that network.",
      "done_when": "A reproducible firmware reference exists and a cold boot reaches the local test mode without attempting radio attachment."
    },
    {
      "path": "/execution",
      "action": "Choose timing, seed, and two local streams from queue capacity and pilot traces rather than arbitrary values.",
      "method": "Run three non-reportable pilots: first a critical periodic stream, then a bounded bulk burst, then their combination. Capture timer release, queue admission, expiry, and terminal events; set duration and repetition count only after every intended path appears and every released item reconciles.",
      "done_when": "The ready file has a concrete duration, seed plan, and stream values supported by pilot traces with no unexplained counter mismatch."
    },
    {
      "path": "/acceptance",
      "action": "Turn the local result into a valid accounting gate rather than a vague smoke test.",
      "method": "Require counter reconciliation, define whether valid local timing permits a critical on-time floor, list the minimal evidence files, and explain any forced expiry or rejection in operator notes. Do not claim network latency in this scenario.",
      "done_when": "The evidence bundle lets another reader recompute each terminal outcome without a broker, dashboard, or host model."
    }
  ]
}


## 11. Stable Baseline — Core Final Set

Baseline adalah anchor physical evidence. Ketiga TODO repository diselesaikan berurutan:

1. **`/setup` — freeze real four-board topology.** Catatan berasal dari repeated power-cycle attachment: label, role, reachability, surveyed channel, placement/orientation, power/cables, source/upstream/ESP-IDF/sdkconfig, dan binary hashes.
2. **`/execution` — choose workload and duration from pilots.** Minimal tiga pilot non-reportable mengevaluasi counter reconciliation, release count, queue bounds, warm-up, measured window, cooldown, seed plan, dan variance. Final repetitions dibekukan sebelum final outcome.
3. **`/acceptance` — predeclare service floor and invalidation.** `counter_reconciliation` wajib, `minimum_critical_on_time_pdr` berasal dari keputusan pra-hasil, dan `negative_case` mendefinisikan invalidity yang dapat dinilai tanpa model.

Final repetitions memakai manifest bytes dan physical identity yang sama. Setiap reboot tetap menghasilkan boot identity baru yang direkam, bukan disembunyikan.


### `experiments/authoring/baseline.jsonc`

Cell berikut adalah salinan persis dari `experiments/authoring/baseline.jsonc` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/baseline.jsonc
{
  // Authoring copy only. Keep the matching ../baseline.json as strict JSON.
  // Do not set state to ready until every null below has a measured or frozen value.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stable-baseline",
  "title": "Stable Thread Baseline",
  "purpose": {
    // Preserve the question. If the question changes, create a new experiment ID.
    "question": "What does normal deadline delivery look like on the physical Thread topology before modelling or policy changes?",
    "comparison": "Repeated static runs with the same workload, no host model, and no remote actuation.",
    "primary_metric": "Reconciled on-time delivery ratio for the critical stream."
  },
  "setup": {
    // Replace with exactly the four physical roles and their labels after repeated attachment works.
    "nodes": null,
    // Choose one surveyed IEEE 802.15.4 channel; do not select it from a favorable result.
    "thread_channel": null,
    // Describe positions, orientation, power path, cables, and the named baseline evidence block.
    "placement": null,
    // Record source revision, ESP-IDF/upstream revision, sdkconfig hash, and binary hashes.
    "firmware_reference": null
  },
  "execution": {
    // Warm-up is excluded from the primary calculation but must be recorded.
    "warmup_s": null,
    // Choose a long enough measured window to capture normal variance and many critical releases.
    "measurement_s": null,
    // Cooldown ends only after final counter and trace collection is requested.
    "cooldown_s": null,
    // Freeze the planned count before examining final baseline results.
    "repetitions": null,
    // Record the deterministic workload seed or an explicitly documented seed plan.
    "seed": null
  },
  "traffic": {
    // Add concrete critical and telemetry streams after pilots. Baseline has no host policy.
    "streams": null
  },
  "scenario": {
    // Stable baseline uses event none and a non-empty target such as "no injected disturbance".
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Use baseline mode, candidate_action none, host_model false, remote_actuation false.
    "mode": null,
    "candidate_action": null,
    // Name the immutable baseline profile even when it cannot actuate; archive its resolved digest.
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    // Must be true for every reportable result.
    "counter_reconciliation": null,
    // Select this floor before final analysis, based on traceable pilot evidence.
    "minimum_critical_on_time_pdr": null,
    // State the failure that invalidates this baseline, for example unreconciled counters.
    "negative_case": null
  },
  "evidence": {
    // Require manifest, versions, topology, raw events, final counters, and operator notes at minimum.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Freeze the real four-board topology only after upstream RCP and border-router attachment are repeatable.",
      "method": "Label every board, survey one channel and one work area, record placement and power path, then archive actual Thread roles and binary identities.",
      "done_when": "The configuration survives power cycling and its placement, roles, and reachability are archived."
    },
    {
      "path": "/execution",
      "action": "Choose stable workload and duration from pilots.",
      "method": "Run at least three pilots, changing one offered-load factor at a time until counters reconcile and queue behavior is bounded.",
      "done_when": "Durations, streams, seed, and repetitions are frozen before reportable baseline runs."
    },
    {
      "path": "/acceptance",
      "action": "Predeclare service floor and invalidation rules.",
      "method": "Require reconciliation, static treatment, and sufficient raw evidence to classify each run.",
      "done_when": "A later model result is not needed to decide whether baseline data are valid."
    }
  ]
}


### `experiments/baseline.json`

Cell berikut adalah salinan persis dari `experiments/baseline.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/baseline.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stable-baseline",
  "title": "Stable Thread Baseline",
  "purpose": {
    "question": "What does normal deadline delivery look like on the physical Thread topology before modelling or policy changes?",
    "comparison": "Repeated static runs with the same workload, no host model, and no remote actuation.",
    "primary_metric": "Reconciled on-time delivery ratio for the critical stream."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Freeze the real four-board topology only after upstream RCP and border-router attachment are repeatable.",
      "method": "Label the S3 gateway, RCP C6, router-capable C6, and low-power C6; survey the ordinary work area; select one 802.15.4 channel; mark board position, orientation, power path, and cables; then record actual Thread roles and binary identities.",
      "done_when": "The same configuration survives power cycling, IPv6 reachability is confirmed, and placement plus role evidence are archived before a reportable run."
    },
    {
      "path": "/execution",
      "action": "Find a stable workload and run length that characterize normal variance without hiding queue behavior.",
      "method": "Use three pilot repetitions with one critical and one telemetry stream. Adjust only one offered-load dimension at a time until the queue remains bounded, all counters reconcile, and the measurement window contains enough critical releases to make a service ratio meaningful.",
      "done_when": "The ready file has frozen stream details, phase durations, seed, and repetition count derived from a traceable pilot decision."
    },
    {
      "path": "/acceptance",
      "action": "Set an engineering service floor and raw-evidence requirement before calibration data is used by any model.",
      "method": "Choose a numerical critical on-time floor stricter than pilot noise, require reconciliation, set static/no-model/no-actuation treatment values, and require manifest, versions, topology, events, final counters, and operator notes in evidence.",
      "done_when": "Every baseline run can be classified complete, invalid, or interrupted without looking at a later model result."
    }
  ]
}


## 12. Held-Out Load Step — Core Shadow Set

`load-step-prediction` membandingkan tiga model pada horizon yang sama; ini bukan controller test.

1. **`/setup` — bind to completed baseline block.** Board, channel, placement, firmware, dan physical block sama. Perubahan apa pun memulai calibration block baru.
2. **`/traffic` — one recoverable bulk step.** Pilot menetapkan step time/duration dan hanya bulk period/payload/burst yang berubah. Critical stream tetap frozen; step harus measurable tetapi recoverable.
3. **`/treatment` — shadow-only.** `mode` prediction, `host_model` true, `remote_actuation` false, `candidate_action` none, dan frozen model/profile identity.
4. Calibration dan held-out membership dideklarasikan sebelum scoring; tidak ada retuning dari held-out plot.
5. Semua model memakai identical horizons dan exclusions. Run yang invalid tetap terlihat di ledger, tidak masuk denominator valid secara diam-diam.

Primary metric tetap held-out P95 prediction error dengan run-aware uncertainty sebagaimana manifest; gate metrics bersifat secondary.


### `experiments/authoring/load-step.jsonc`

Cell berikut adalah salinan persis dari `experiments/authoring/load-step.jsonc` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/load-step.jsonc
{
  // This authoring copy plans a held-out prediction experiment, never a controller test.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "load-step-prediction",
  "title": "Held-Out Load-Step Prediction",
  "purpose": {
    "question": "Can a cross-layer model predict critical deadline degradation when a known bulk load step is introduced?",
    "comparison": "Naive, network-only, and cross-layer predictions on identical horizons from a load pattern not used to tune the models or gate.",
    "primary_metric": "Held-out P95 prediction error with run-aware uncertainty; gate trust metrics remain secondary."
  },
  "setup": {
    // Copy the completed stable baseline topology exactly and cite its evidence-bundle identifier.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    // Keep the same phase structure as the baseline unless a predeclared reason requires change.
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Freeze the critical stream; alter only predeclared bulk period, payload, or burst to form the load step.
    "streams": null
  },
  "scenario": {
    // Use load_step after warm-up and inside the measured window; identify the affected bulk stream.
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Required: prediction mode, host_model true, remote_actuation false, candidate_action none.
    "mode": null,
    "candidate_action": null,
    // Name the calibration/profile identity frozen before held-out scoring.
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    "counter_reconciliation": null,
    // Carry forward the baseline critical-service floor rather than loosening it after the step.
    "minimum_critical_on_time_pdr": null,
    // Define an invalid case such as model code or physical topology changing during the held-out block.
    "negative_case": null
  },
  "evidence": {
    // Include calibration/held-out split, all three predictions, calibrated-region and observation-integrity status, raw traces, and model revision.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Bind this run to one completed baseline block.",
      "method": "Reuse board identities, channel, placement, and firmware; declare a new calibration block if any one changes.",
      "done_when": "No network or firmware change is hidden inside the prediction condition."
    },
    {
      "path": "/traffic",
      "action": "Create one recoverable bulk load step.",
      "method": "Use pilots outside the final set until critical behavior changes measurably but queues and radios recover.",
      "done_when": "Step time, duration, stream values, seed, and repetitions are frozen before held-out analysis."
    },
    {
      "path": "/treatment",
      "action": "Keep the run strictly shadow-only.",
      "method": "Score all three models on identical future horizons and prohibit model or gate tuning from held-out observations.",
      "done_when": "No remote policy can be issued in this condition."
    }
  ]
}


### `experiments/load-step.json`

Cell berikut adalah salinan persis dari `experiments/load-step.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/load-step.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "load-step-prediction",
  "title": "Held-Out Load-Step Prediction",
  "purpose": {
    "question": "Can a cross-layer model predict critical deadline degradation when a known bulk load step is introduced?",
    "comparison": "Naive, network-only, and cross-layer predictions on identical horizons from a load pattern not used to tune the models or gate.",
    "primary_metric": "Held-out P95 prediction error with run-aware uncertainty; gate trust metrics remain secondary."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Bind this held-out condition to a completed stable-baseline block before changing workload shape.",
      "method": "Copy the validated physical board identities, channel, placement, and firmware reference from one archived baseline block. If any of those facts change, declare a new calibration block instead of calling the traces held out.",
      "done_when": "The ready file cites a specific baseline evidence bundle and no network or firmware change is hidden inside the prediction condition."
    },
    {
      "path": "/traffic",
      "action": "Design one recoverable bulk load step while preserving the frozen critical stream.",
      "method": "Use pilots outside the final set to increase only bulk period, payload, or burst until critical behavior changes measurably but the queue and radio recover. Place the step after warm-up, keep it inside measurement, and define its target and duration before observing any held-out result.",
      "done_when": "The ready file fixes all streams, event time, duration, seed, and repetition count, and pilot logs show a visible but recoverable disturbance."
    },
    {
      "path": "/treatment",
      "action": "Keep this experiment as a shadow-model prediction test, not a disguised controller test.",
      "method": "Set prediction mode, enable the host model, disable remote actuation, choose a critical service floor from the baseline, and require raw traces plus a prediction report that identifies calibration data, model revision, prediction horizon, all three models, calibrated-region status, and observation-integrity status.",
      "done_when": "No held-out observation is used to tune model parameters or gate thresholds and the report scores all models on exactly the same horizons."
    }
  ]
}


## 13. Stale Observation — Conditional Safety Final Set

Pair ini eligible hanya jika normal finite command path dan seluruh Week-5 gate sudah terbukti. Remote control tidak diaktifkan demi menyelesaikan calendar.

1. **`/setup` — prove only observations stop.** Application publication gate menjadi target; local endpoint traffic dan Thread radio tetap normal.
2. **`/scenario` — exceed frozen freshness limit.** Normal observation age diukur, `maximum_observation_age_ms` berasal dari resolved profile, pause dimulai setelah warm-up, dan expected latest fallback time ditulis sebelum run.
3. **`/treatment` — finite bounded bulk action only.** Gateway/endpoint audit mencatat accept, reject, expiry, withdrawal, fallback, ABSTAIN→OBSERVE, serta requalification atau continued abstention.
4. Run tidak valid jika RF impairment, restart, load-step baru, atau perubahan topology ikut terjadi.
5. Kegagalan mencapai fallback deadline adalah hasil safety yang dilaporkan, bukan alasan menghapus run.

Jika eligibility tidak lengkap, manifest tetap template dan ledger menulis `not run: entry gate reason`.


### `experiments/authoring/stale-observation.jsonc`

Cell berikut adalah salinan persis dari `experiments/authoring/stale-observation.jsonc` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/stale-observation.jsonc
{
  // This authoring copy plans a safety fallback test without disturbing RF or Thread routing.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stale-observation-fallback",
  "title": "Stale Observation Safety Fallback",
  "purpose": {
    "question": "Does the fidelity gate abstain and restore the local safe policy when host observations become stale?",
    "comparison": "Gated control before the observation pause versus the gateway state during and after the pause.",
    "primary_metric": "Time from stale-observation condition to recorded local fallback."
  },
  "setup": {
    // Reuse a fully validated physical baseline; identify the exact gateway-to-host publication path to pause.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Preserve the critical traffic profile while allowing one bounded bulk control surface.
    "streams": null
  },
  "scenario": {
    // Use observation_pause. Duration must exceed maximum_observation_age_ms with a recorded margin.
    "event": null,
    "at_s": null,
    "duration_s": null,
    // Name the publication gate or adapter that stops observations, not the Thread radio.
    "target": null
  },
  "treatment": {
    // Required after normal command path exists: gated_control, bulk_rate_reduce, host model and remote actuation enabled.
    "mode": null,
    "candidate_action": null,
    // Name a resolved profile with finite TTL and documented freshness/hysteresis limits.
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    "counter_reconciliation": null,
    // The same critical floor applies before and after fallback.
    "minimum_critical_on_time_pdr": null,
    // State the expected invalid condition, for example fallback not reached by the predeclared deadline.
    "negative_case": null
  },
  "evidence": {
    // Require newest observation, gate transition, command audit, fallback record, counters, and every recovery/requalification window.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Prove that only observations stop.",
      "method": "Add an application-level publication gate and dry-run it while physical endpoint traffic remains visible locally.",
      "done_when": "The pause boundary is traceable and no RF impairment is used."
    },
    {
      "path": "/scenario",
      "action": "Schedule stale data beyond the frozen freshness limit.",
      "method": "Measure normal observation age, record the profile limit, choose pause timing after warm-up, and use one injected fault.",
      "done_when": "Expected latest fallback time is recorded before the run."
    },
    {
      "path": "/treatment",
      "action": "Enable only a finite bounded bulk action.",
      "method": "Require gateway and endpoint acknowledgements for accept, reject, expiry, and fallback.",
      "done_when": "The evidence can reconstruct withdrawal, ABSTAIN-to-OBSERVE recovery, and either full requalification or continued abstention."
    }
  ]
}


### `experiments/stale-observation.json`

Cell berikut adalah salinan persis dari `experiments/stale-observation.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/stale-observation.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "stale-observation-fallback",
  "title": "Stale Observation Safety Fallback",
  "purpose": {
    "question": "Does the fidelity gate abstain and restore the local safe policy when host observations become stale?",
    "comparison": "Gated control before the observation pause versus the gateway state during and after the pause.",
    "primary_metric": "Time from stale-observation condition to recorded local fallback."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Reuse a validated stable Thread block and identify the precise gateway-to-host observation path that will be paused.",
      "method": "Freeze board roles, placement, channel, binaries, and normal observation cadence. Add an application-level publication gate at the gateway or host adapter; do not jam RF, stop endpoints, or modify Thread routing for this test.",
      "done_when": "A dry run shows physical endpoint traffic continues while the chosen observation stream stops at a traceable boundary."
    },
    {
      "path": "/scenario",
      "action": "Schedule a stale-observation interval that is longer than the calibrated freshness limit and short enough to observe recovery.",
      "method": "Measure normal observation age first, freeze the gate's freshness threshold in the versioned control profile, then choose pause time after warm-up and duration with margin beyond that threshold. Use only one fault in the run.",
      "done_when": "The ready manifest identifies observation_pause, its monotonic time, duration, and target; operator notes state the expected latest fallback time."
    },
    {
      "path": "/treatment",
      "action": "Enable only one finite, locally bounded action before proving its withdrawal.",
      "method": "Use bulk-rate reduction only, turn on the host model and remote actuation, set a finite command TTL in the control profile, keep a compiled safe static policy, and require gateway and endpoint acknowledgements for acceptance, rejection, expiry, and fallback.",
      "done_when": "The evidence includes last accepted observation, gate transition, command or expiry decision, local fallback, restored observations, the full requalification-window sequence, post-fallback counters, and a critical-service floor that the fallback must preserve."
    }
  ]
}


## 14. Restart/Replay — Conditional Safety Final Set

Safety case ini menggunakan normal authenticated gateway path; tidak boleh memanggil endpoint apply function secara langsung.

1. **`/setup` — establish normal persist-before-apply evidence.** Satu accepted current command, replay-record commit, acknowledgement, run/coordinator/endpoint boot identity, epoch, TTL, profile, dan key identity diarsipkan.
2. **`/scenario` — one restart, finite replay matrix.** Endpoint target direstart; durable state reload diverifikasi; old epoch, expired, wrong run, wrong authority boot, corrupt tag, exact replay, serta missing/corrupt durable-record fixture diuji sesuai daftar pra-run.
3. **`/acceptance` — zero invalid applications.** Setiap attempt mempunyai expected reason, actual reason, durable-state transition, local fallback, dan fresh commissioning path.
4. Critical workload dan topology tetap baseline; tidak ada load step atau fault kedua.
5. Satu invalid application berarti safety invariant gagal dan remote-off tetap berlaku.

Jika full command/state evidence tidak tersedia, verdict-nya non-evaluable—bukan pass.


### `experiments/authoring/restart-replay.jsonc`

Cell berikut adalah salinan persis dari `experiments/authoring/restart-replay.jsonc` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/restart-replay.jsonc
{
  // This authoring copy tests command freshness through the normal gateway path.
  // It must never bypass authentication or call an endpoint apply function directly.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "restart-replay-safety",
  "title": "Restart And Command Replay Safety",
  "purpose": {
    "question": "Can a restarting endpoint reload durable replay state, reject stale or replayed policy epochs, and remain safe when that state is unavailable?",
    "comparison": "Command acceptance before restart, rejection after restart with valid durable state, and fail-safe behavior with missing or corrupt state.",
    "primary_metric": "Number of invalid policy applications after the deliberate restart."
  },
  "setup": {
    // Use a topology where one global authenticated finite-TTL command was durably recorded before apply and acknowledged.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    // Include stable operation, restart, reattachment, replay attempts, and fresh synchronization.
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Preserve the validated stable workload; do not add a load step to this safety case.
    "streams": null
  },
  "scenario": {
    // Use endpoint_restart and name the physical endpoint label that will be restarted.
    "event": null,
    "at_s": null,
    // Duration must cover reattachment and fresh synchronization, not merely power-cycle time.
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Safety mode may use only the already validated bounded bulk-rate action.
    "mode": null,
    "candidate_action": null,
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    // Reconciliation still applies; a safety test is not exempt from accounting.
    "counter_reconciliation": null,
    "minimum_critical_on_time_pdr": null,
    // Set the concrete negative outcome: any stale, replayed, or wrong-run policy application.
    "negative_case": null
  },
  "evidence": {
    // Require command audit with active run, coordinator plus old/new endpoint boot IDs, durable replay state, epoch, TTL, and per-attempt reason.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Establish normal global-command and persist-before-apply evidence before injecting a restart.",
      "method": "Commission a unique run and archive an accepted current command, replay-record commit, acknowledgement, endpoint boot ID, epoch, and TTL.",
      "done_when": "There is a pre-restart command and durable-state baseline against which rejection behavior can be interpreted."
    },
    {
      "path": "/scenario",
      "action": "Define one restart and a finite set of replay attempts.",
      "method": "Restart the selected endpoint, verify replay-record reload, then submit old-epoch, expired, wrong-run, wrong-authority-boot, corrupted-tag, and exact replay commands through the normal gateway path. Separately use a missing-or-corrupt-record fixture.",
      "done_when": "Operator notes list durable-state fixtures, exact invalid inputs, and expected rejection reasons."
    },
    {
      "path": "/acceptance",
      "action": "Predeclare zero invalid applications.",
      "method": "Require persist-before-apply, local fallback when replay state is unavailable, explicit new-run commissioning, and complete command/state evidence.",
      "done_when": "A reviewer can reconstruct every attempted command and durable-state transition."
    }
  ]
}


### `experiments/restart-replay.json`

Cell berikut adalah salinan persis dari `experiments/restart-replay.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/restart-replay.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "restart-replay-safety",
  "title": "Restart And Command Replay Safety",
  "purpose": {
    "question": "Can a restarting endpoint reload durable replay state, reject stale or replayed policy epochs, and remain safe when that state is unavailable?",
    "comparison": "Command acceptance before restart, rejection after restart with valid durable state, and fail-safe behavior with missing or corrupt state.",
    "primary_metric": "Number of invalid policy applications after the deliberate restart."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Prove the normal global-command path, durable replay record, and boot identity observability before injecting a restart.",
      "method": "Reuse a valid safety-capable Thread topology, select the endpoint to restart, commission a unique run, and verify that one current authenticated finite-TTL global policy is durably recorded before it is applied and acknowledged.",
      "done_when": "A pre-restart trace contains active run identity, coordinator and endpoint boot IDs, accepted epoch, replay-record commit result, TTL, and acknowledgement reason."
    },
    {
      "path": "/scenario",
      "action": "Run one controlled restart and explicitly enumerate the replay attempts that follow it.",
      "method": "Choose endpoint_restart time after stable operation, capture duration through reattachment, verify the durable record reload, then send the ordinary gateway path an old epoch, an expired command, a wrong-run command, a wrong-authority-boot command, a corrupted tag, and an exact replay. Use a separate missing-or-corrupt-record fixture; never bypass authentication or call private apply functions directly.",
      "done_when": "The ready file fixes restart time and target, while operator notes list the replay-record fixtures, exact invalid command cases, and expected reject reasons."
    },
    {
      "path": "/acceptance",
      "action": "Make zero invalid policy applications the safety outcome and preserve complete command evidence.",
      "method": "Set safety treatment, select only the bounded bulk-rate action, require persist-before-apply, require the endpoint to keep its safe policy when replay state is unavailable, preserve critical service floor and counter reconciliation, and require command-audit, replay-state, raw-event, final-counter, version, and run-status artifacts.",
      "done_when": "A reviewer can reconstruct every attempted command and durable-state transition, confirm that stale, replayed, or wrong-run input changed no endpoint policy, and confirm that missing replay state required a new run."
    }
  ]
}


## 15. Topology Shift — Conditional Depth, Bukan Core Queue

`topology-shift` hanya dijalankan bila sudah stabil pada 19 October, core final repetitions selesai, dan tidak mengambil waktu evidence repair. Notebook mempertahankan pair repository agar batasnya jelas, bukan menjadwalkannya sebagai kewajiban.

1. Kedua posisi satu endpoint harus dapat direka ulang dari marks, dimensions, orientation, obstacles, cables, photos, role, RLOC16, parent, partition, dan link state.
2. Hanya endpoint yang dinamai bergerak; channel, gateway/RCP/endpoint lain, workload, firmware, dan model tetap frozen.
3. Support envelope, error tolerance, gate thresholds, dan minimum useful in-domain trust fraction sudah frozen.
4. Outcome retained fidelity **atau** abstention sama-sama sah; plot tidak boleh memilih rule setelah hasil.
5. Bila syarat tidak terpenuhi, tulis `not run: conditional depth cut`. Jangan membuat placeholder result.

Ablation mengikuti aturan yang sama dan tidak memiliki manifest baru di repo. Karena itu notebook tidak mengarang `ablation` manifest.


### `experiments/authoring/topology-shift.jsonc`

Cell berikut adalah salinan persis dari `experiments/authoring/topology-shift.jsonc` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/topology-shift.jsonc
{
  // This authoring copy plans one controlled physical placement change and no additional injected fault.
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "topology-shift",
  "title": "Controlled Placement And Link-Context Shift",
  "purpose": {
    "question": "Does a frozen calibration-envelope-aware gate avoid false trust when one endpoint enters a held-out physical context?",
    "comparison": "Always-trust, residual/freshness-only, and calibrated-region-aware decisions on identical predeclared horizons before and after one placement change.",
    "primary_metric": "False-trust rate and trusted-horizon fraction, with abstention and requalification latency."
  },
  "setup": {
    // Describe reproducible before/after locations for one endpoint: distance, height, orientation, obstacles, cables, and photos.
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    // Measurement must contain stable pre-shift, movement, and enough post-shift recovery/degradation time.
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": {
    // Reuse completed baseline streams unchanged so placement is the sole intended disturbance.
    "streams": null
  },
  "scenario": {
    // Use topology_shift and name exactly one movable endpoint label.
    "event": null,
    "at_s": null,
    "duration_s": null,
    "target": null
  },
  "treatment": {
    // Start as prediction/shadow mode with remote actuation false unless the primary safety chain is already complete.
    "mode": null,
    "candidate_action": null,
    "control_profile": null,
    "host_model": null,
    "remote_actuation": null
  },
  "acceptance": {
    "counter_reconciliation": null,
    "minimum_critical_on_time_pdr": null,
    // Define invalid case such as unrecorded second placement change or unplanned restart/load change.
    "negative_case": null
  },
  "evidence": {
    // Require before/after photos, placement measurements, role/RLOC16/parent/partition/link diagnostics, per-horizon support/integrity state, gate report, and raw events.
    "required_artifacts": null,
    "operator_notes_required": null,
    "topology_photo_required": null
  },
  "_todo": [
    {
      "path": "/setup",
      "action": "Make both physical positions reproducible.",
      "method": "Mark and photograph locations; freeze gateway/RCP/other endpoint positions and record role, RLOC16, parent, partition, and link state at each position. Use topology-shift wording only if those observations support it.",
      "done_when": "A second operator can recreate both configurations."
    },
    {
      "path": "/scenario",
      "action": "Inject one documented shift.",
      "method": "Move only the named endpoint after warm-up and reserve a post-shift interval; do not combine restart or load step.",
      "done_when": "Placement is the sole intended cause of change."
    },
    {
      "path": "/treatment",
      "action": "Predeclare acceptable fidelity outcomes.",
      "method": "Freeze the support envelope, gate limits, error tolerance, and minimum useful in-domain trust fraction; permit a conclusion of either retained fidelity or abstention.",
      "done_when": "The graph cannot choose the rule after the result is known."
    }
  ]
}


### `experiments/topology-shift.json`

Cell berikut adalah salinan persis dari `experiments/topology-shift.json` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/topology-shift.json
{
  "schema_version": "2.0",
  "state": "template",
  "experiment_id": "topology-shift",
  "title": "Controlled Placement And Link-Context Shift",
  "purpose": {
    "question": "Does a frozen calibration-envelope-aware gate avoid false trust when one endpoint enters a held-out physical context?",
    "comparison": "Always-trust, residual/freshness-only, and calibrated-region-aware decisions on identical predeclared horizons before and after one placement change.",
    "primary_metric": "False-trust rate and trusted-horizon fraction, with abstention and requalification latency."
  },
  "setup": {
    "nodes": null,
    "thread_channel": null,
    "placement": null,
    "firmware_reference": null
  },
  "execution": {
    "warmup_s": null,
    "measurement_s": null,
    "cooldown_s": null,
    "repetitions": null,
    "seed": null
  },
  "traffic": { "streams": null },
  "scenario": { "event": null, "at_s": null, "duration_s": null, "target": null },
  "treatment": { "mode": null, "candidate_action": null, "control_profile": null, "host_model": null, "remote_actuation": null },
  "acceptance": { "counter_reconciliation": null, "minimum_critical_on_time_pdr": null, "negative_case": null },
  "evidence": { "required_artifacts": null, "operator_notes_required": null, "topology_photo_required": null },
  "_todo": [
    {
      "path": "/setup",
      "action": "Define two repeatable endpoint locations before collecting a topology-shift trace.",
      "method": "Mark the before and after position of one chosen endpoint with distance, height, orientation, obstacles, cable routing, and photographs. Freeze gateway, RCP, and other endpoint positions; record actual Thread role, RLOC16, parent, partition, RSSI/link quality, and available MAC/MLE evidence at both positions. Call it a topology shift only if topology evidence actually changes.",
      "done_when": "A second operator can recreate both positions and distinguish a planned shift from accidental movement."
    },
    {
      "path": "/scenario",
      "action": "Schedule exactly one physical shift after a stable baseline interval.",
      "method": "Reuse a completed baseline traffic profile, set topology_shift target to the selected endpoint, choose the move time after warm-up, and reserve a post-shift interval long enough to observe reattachment or stable degradation. Do not introduce a load step or restart in the same run.",
      "done_when": "The ready file defines event time, duration, target, and unchanged traffic so placement is the sole intended disturbance."
    },
    {
      "path": "/treatment",
      "action": "Predeclare what counts as a correct fidelity response.",
      "method": "Run as a host-model shadow with remote actuation disabled unless the primary safety chain is already proven. Freeze the support-envelope rule, gate limits, minimum useful in-domain trust fraction, error tolerance, service floor, and evidence list before target runs.",
      "done_when": "The report includes false trust, trusted-horizon fraction, selective error, abstention latency, and requalification latency and can honestly conclude either retained fidelity or abstention under the frozen rules."
    }
  ]
}


## 16. Hardware Preflight — Same Four Boards

Tidak ada pembelian atau topology baru. Physical set tetap:

- ESP32-S3 gateway;
- ESP32-C6 RCP;
- ESP32-C6 endpoint A;
- ESP32-C6 endpoint B;
- host, powered USB hub, known-good data cables, stable power, dan access point yang sudah dipakai pada baseline.

Preflight dicatat sekali per physical block:

1. Cocokkan label fisik, serial/port mapping, board revision, cable, hub port, dan power path.
2. Verifikasi source revision, ESP-IDF/upstream revision, sdkconfig digest, serta SHA-256 setiap binary sebelum flash.
3. Flash image frozen; simpan complete build/flash log. Jangan rebuild “yang sama” tanpa membandingkan hash.
4. Cold boot seluruh board dalam urutan yang telah divalidasi; catat boot IDs, Thread role, partition, parent/RLOC16, IPv6, channel, dan reachability.
5. Cocokkan placement marks, orientation, obstacles, access-point position, dan foto topology.
6. Cek host disk/time/timezone, monotonic clock source, broker/process state, recorder write access, dan free space.
7. Dry-run manifest admission serta output directory collision rejection tanpa memulai final measurement.
8. Jalankan short non-reportable smoke observation. Jika identity atau reconciliation gagal, final queue belum dimulai.

Nomor port, role, alamat, dan hash tidak diisi lebih awal karena semuanya harus dibaca dari hardware/evidence block aktual.


## 17. One Physical Repetition

Satu repetition mempunyai awal dan akhir yang dapat diaudit:

1. Reserve unique run identity sebelum traffic mulai.
2. Copy strict manifest bytes, simpan digest, source/binary/tool/profile identities, operator, physical block, dan intended repetition index.
3. Buka recorder dan verifikasi append-only stream aktif.
4. Jalankan warm-up tanpa memasukkan samples ke primary measured window; event boundary tetap direkam.
5. Mulai measured window pada predeclared boundary. Tidak ada tuning, port swapping, manual queue clearing, atau replacement run.
6. Inject hanya scenario event dari manifest pada exact boundary; baseline memakai no injected disturbance.
7. Akhiri measured window dan masuk cooldown sambil meminta final counters, trace, topology/role state, model/gate/command audit yang eligible.
8. Reconcile released versus terminal events dan aggregate counters.
9. Finalize tepat sekali sebagai complete, invalid, atau interrupted dengan reason vocabulary.
10. Hash/close evidence, update run ledger, lalu backup read-only sebelum repetition berikutnya.

Operator note berisi observasi faktual—restart tak terencana, cable movement, role change, pause timing, error—bukan interpretasi hasil. Tidak ada run yang dihapus karena angkanya buruk.


### Run Ledger

Ledger memakai identitas yang benar-benar tersedia dari manifest/evidence. Satu row per attempt, termasuk pilot, invalid, dan interrupted.

| Human label | Sumber nilai | Contoh bentuk |
|---|---|---|
| Run identity | reserved run/coordinator record | `<run id>` |
| Experiment | strict `experiment_id` | `stable-baseline` |
| Manifest digest | archived bytes | `<64 hex>` |
| Physical block | topology/equipment record | `block-…` |
| Repetition | frozen plan + attempt order | `1 of N` |
| Start/end | host timestamp + timezone | ISO-8601 |
| Source/binaries | Git + SHA-256 evidence | commit/hash |
| Board boots/roles | device logs | label → boot/role |
| Terminal status | recorder vocabulary | `complete` / `invalid` / `interrupted` |
| Reason | predefined reason + factual note | text |
| Reconciliation | reproduction/audit | example `pass` or exact counters |
| Eligible analysis | admission decision | `yes` / `no: reason` |
| Evidence location | immutable bundle reference | path or archive ID |

`N` berasal dari frozen `execution.repetitions`; notebook tidak memilih nilainya.


## 18. Raw Immutability dan Identity Rollover

Raw evidence tidak diedit, dinormalisasi, digabung, atau “dibersihkan.” Parsing corrections dilakukan pada code/config baru sambil mempertahankan original bytes dan digest.

Identity rollover wajib ketika salah satu berubah:

- behavior source atau compiler/toolchain yang memengaruhi binary;
- firmware, sdkconfig, binary hash, upstream RCP/border-router image;
- schema/manifest bytes atau experiment plan;
- board replacement, role assignment, channel, placement, cable/power path yang menjadi controlled setup;
- traffic, seed plan, duration, repetition plan;
- model features/parameters, calibration membership, horizon/metric;
- fidelity gate, control profile, command identity/key, or safety policy;
- recorder/reproduction semantics.

Run lama tetap di ledger. Jika change hanya memperbaiki parser tanpa mengubah raw data, derived output memakai analysis identity baru dan menyebut input digests; ia tidak menimpa output lama.


## 19. Statistical Freeze

Primary analysis mengikuti contract yang sudah dideklarasikan:

1. Unit physical replication adalah run, bukan packet, event, atau bootstrap sample.
2. Calibration/held-out split dibekukan sebelum held-out scoring.
3. Naive, network-only, dan cross-layer model memakai target/horizon/exclusion yang identik.
4. Primary metric adalah relative P95 prediction error pada deadline-delivery ratio; zero/near-zero denominator mempunyai rule sebelum scoring.
5. Uncertainty berasal dari run-level summaries atau whole-run cluster bootstrap. Within-run block bootstrap hanya menjawab paired temporal uncertainty.
6. Prediction interval coverage memakai interval frozen Phase 4; komentar `+/- 2 sigma` pada scaffold diselaraskan ke contract tersebut, bukan menjadi keputusan baru.
7. Missing, stale, unreconciled, atau out-of-support horizons dilaporkan sebagai status; tidak hilang dari denominator tanpa reason.
8. Multiple secondary metrics tidak dipakai untuk menggantikan primary metric yang gagal.
9. Ablation hanya menjelaskan contribution setelah primary result frozen.
10. Semua random seeds, library versions, configuration, input digests, dan code revision ikut output.

Report menyajikan distribution/uncertainty dan seluruh valid/invalid/interrupted counts, bukan best run.


## 20. Safety-Result Freeze

Untuk Week-5 evidence yang eligible, reproduction merangkum:

- gate state dan reason versus time;
- observation age dan integrity status;
- support-envelope status;
- trust fraction dan false-trust definition;
- abstention dan requalification latency;
- `P[2][2]` sesuai frozen estimator contract;
- proposal, authenticated command, gateway decision, endpoint decisions/acks;
- command TTL/expiry/withdrawal/fallback;
- restart identity, durable replay state, attempt matrix, rejection reason;
- critical-service counters sebelum, selama, dan setelah action/fault.

Claim safety dibuat per invariant. Contoh: bukti stale fallback tidak otomatis membuktikan replay rejection. Jika satu invariant gagal, conclusion menyebut kegagalan itu dan `remote_actuation` tetap disabled. Tidak ada aggregate “safe” verdict yang menutupi trace parsial.


## 21. Clean Reproduction Rehearsal

Seorang reproducer kedua memakai clean clone dan evidence copy read-only:

1. Checkout exact source revision.
2. Verifikasi repository status, submodule/upstream reference bila ada, toolchain, serta analysis dependencies frozen.
3. Build/test dari clean directory dan konfirmasi tujuh C tests aktif, nol skipped.
4. Verifikasi evidence archive digests sebelum parsing.
5. Jalankan reproduction menggunakan invocation contract yang ditampilkan oleh script.
6. Catat stdout, stderr, exit code, wall time, environment, dan generated-output digests.
7. Bandingkan metric table, verdict, run counts, exclusions, reconciliation counters, dan figure data dengan frozen reference.
8. Ulangi deliberate negative fixture; script harus gagal pada digest mismatch atau lifecycle inconsistency.
9. Reproducer menulis ambiguity yang ditemui tanpa dibantu oleh state lokal pembuat.
10. Defect diperbaiki hanya bila evidence semantics tidak berubah; jika berubah, analysis identity/version baru digunakan.

Rehearsal lulus bila hasil dan failure behavior dapat dijelaskan dari archived inputs. Kesamaan screenshot saja tidak cukup.


## 22. Git dan Public/Private Evidence Boundary

Repository saat ini mengabaikan raw/results/build/secrets. Ini melindungi data dan key, tetapi berarti push source tidak sama dengan mengarsipkan evidence.

Aturan handoff:

1. Secret, command key, Wi-Fi credential, dan device-specific credential tidak masuk Git atau public evidence.
2. Raw/private evidence disimpan di lokasi terkontrol dengan checksum, backup, retention, dan access record.
3. Public result hanya memuat artefak yang sudah ditinjau untuk credential, MAC/IP/device identifiers, wajah/lokasi, dan sensitive metadata.
4. Derived public table/figure selalu menunjuk input evidence identity dan analysis revision.
5. File ignored tidak pernah dianggap “sudah dibackup” hanya karena working tree clean.
6. Line-ending rules dipertahankan agar manifest/source bytes dan digest tidak berubah lintas platform.
7. Sebelum commit: `git status`, staged diff, accidental-large-file scan, secret scan, notebook execution/output scan, link check, dan clean-clone rehearsal.


### `.gitignore`

Cell berikut adalah salinan persis dari `.gitignore` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/gitignore
# Native host build
/build/
/out/
CMakeUserPresets.json

# ESP-IDF outputs
firmware/**/build/
firmware/**/sdkconfig
firmware/**/sdkconfig.old
firmware/**/managed_components/
dependencies.lock

# Experiment data: keep manifests, not bulky or machine-specific results
results/raw/
results/private/
# Do not ignore experiments/*.json: template and ready manifests are evidence.

# Credentials and local network details
secrets/
.env
.env.*
!.env.example
*.pem
*.key
*.crt
*.p12
*.pfx
*.jks

# Tools and editors
.cache/
.vscode/
.idea/
compile_commands.json
*.swp
*~

# Operating systems
.DS_Store
Thumbs.db
Desktop.ini


### `.gitattributes`

Cell berikut adalah salinan persis dari `.gitattributes` pada snapshot notebook. Komentar TODO dan nilai kosong tetap dibiarkan apa adanya; penyelesaiannya terjadi di repository, bukan di salinan ini.


In [ ]:
%%writefile /content/cldt_scratch/gitattributes
* text=auto eol=lf

*.c text eol=lf
*.h text eol=lf
*.md text eol=lf
*.json text eol=lf
*.jsonc text eol=lf
*.yml text eol=lf
*.yaml text eol=lf
*.csv text eol=lf
*.txt text eol=lf
CMakeLists.txt text eol=lf
*.defaults text eol=lf
*.projbuild text eol=lf
*.cff text eol=lf

# Linguist overrides: keep GitHub language classification accurate.
experiments/*.json linguist-vendored
experiments/authoring/*.jsonc linguist-vendored
schemas/*.json linguist-vendored


## 23. Documentation, Limitations, dan Presentation

Dokumentasi diperbarui setelah evidence freeze, bukan untuk mengantisipasi keberhasilan. `README.md` tetap landing page; `EXPERIMENTS.md` tetap methodology owner. Tidak ada nama final-report atau deck baru yang dikarang oleh notebook.

Urutan narasi presentation:

1. Pertanyaan: apakah bounded cross-layer shadow lebih valid daripada baseline pada held-out Thread load, dan kapan control harus abstain?
2. Scope: satu topology kecil, ESP32 family, OpenThread/ESP-IDF, satu physical environment, satu finite bulk action.
3. Contract: manifest → identity → raw events → reconciliation → calibration/held-out → model score → gate/safety trace.
4. Hardware dan workload: hanya configuration yang benar-benar dijalankan.
5. Data quality: run ledger, valid/invalid/interrupted counts, counter reconciliation.
6. Primary result: tiga model pada identical horizons beserta uncertainty.
7. Safety result: stale fallback dan restart/replay hanya jika eligible.
8. Negative/inconclusive evidence: ditampilkan langsung, bukan dipindah ke catatan kaki.
9. Limitations: external validity, sample size, RF context, clock/measurement limits, missing conditional depth.
10. Reproduction: exact source revision, evidence identity, command, dependency/tool versions, dan observed exit behavior.
11. Future work: context breadth, hardware/stack diversity, SMP/power/dashboard hanya sebagai projection.

Diagram architecture atau plot hanya masuk bila mempermudah hubungan yang benar-benar dibuktikan. Tidak ada decorative metric, fabricated screenshot, atau angka placeholder.


### Limitation Register

Setiap limitation menjawab empat hal: apa batasnya, claim mana yang terpengaruh, evidence yang menunjukkan batas itu, dan apa yang diperlukan untuk memperluas validity.

| Batas | Dampak claim |
|---|---|
| Satu topology dan satu physical room/block | Tidak mewakili semua Thread deployment atau RF context |
| ESP32-S3/C6 dan satu OpenThread/ESP-IDF line | Tidak otomatis berlaku pada hardware/stack lain |
| Repetition count dan measurement duration terbatas | Uncertainty dilaporkan; absence of detected effect bukan proof of no effect |
| Application/host timestamps dan clock model | Tidak sama dengan PHY-ground-truth latency |
| Satu finite bulk-rate action | Tidak mendukung klaim general controller atau multi-action optimization |
| Conditional topology/ablation mungkin tidak dijalankan | Generalization/contribution depth tetap terbatas |
| Remote CI availability terpisah dari local evidence | Workflow yang tidak start bukan code pass atau code fail |

External validity yang sempit bukan defect yang harus ditutupi; ia menentukan kalimat claim yang sah.


## 24. Failure Matrix dan Recovery yang Sah

| Failure | Status run/result | Tindakan yang sah |
|---|---|---|
| Build atau active unit test gagal | campaign blocked | Perbaiki owner, rerun tests; identity baru bila behavior berubah |
| Test masih return 77 | verification incomplete | Tidak sebut green; implement assertions |
| Manifest template/null/TODO tersisa | not admitted | Selesaikan dari evidence/plan sebelum run |
| Existing output/run directory | admission failure | Reserve identity baru; jangan overwrite |
| Digest mismatch | non-evaluable | Pertahankan bytes, telusuri provenance |
| Malformed/blank/trailing NDJSON | invalid/non-evaluable sesuai contract | Fail nonzero; jangan drop line |
| Lifecycle reconciliation gagal | invalid | Laporkan counters dan raw evidence |
| Unplanned reboot/role/topology change | interrupted/invalid sesuai rule | Catat; jangan replacement tersembunyi |
| Model misses acceptance | valid negative | Freeze dan laporkan |
| Uncertainty terlalu lebar | inconclusive | Laporkan; jangan post-hoc menambah favorable runs |
| Fallback/replay invariant gagal | valid safety failure | Remote-off; new block hanya setelah defect repair |
| Conditional depth kehabisan waktu | not run | Potong tanpa mengganggu core |
| GitHub job tidak start | CI unavailable | Simpan annotation; jangan klaim pass |

Recovery tidak pernah menghapus run lama, mengubah acceptance setelah hasil, atau mencampur identity blocks.


## Phase-6 Delivery Checklist
### Closure sebelum Handoff 27 October

**Source dan verification**

- [ ] Exact source revision/tag dicatat dan working tree final bersih.
- [ ] Host/common configure-build selesai dari clean directory.
- [ ] Tujuh existing C tests mempunyai assertions aktif; nol skipped dan nol failed.
- [ ] Firmware builds/flashes yang dipakai mempunyai toolchain/sdkconfig/binary hashes.
- [ ] Remote CI status dilaporkan apa adanya, termasuk job yang tidak mulai.

**Manifests dan identity**

- [ ] Strict manifest setiap run valid schema, `state: ready`, `_todo: []`.
- [ ] Authoring/strict identity konsisten dan strict bytes mempunyai SHA-256.
- [ ] Repetition/seed/duration/traffic/acceptance frozen sebelum outcome.
- [ ] Setiap source/binary/topology/model/profile change memulai identity block baru.
- [ ] Conditional experiment ditandai eligible atau `not run` dengan reason.

**Hardware dan raw evidence**

- [ ] Empat board, role, port, boot, channel, placement, power/cable path tercatat.
- [ ] Setiap attempt—pilot/final/invalid/interrupted—mempunyai ledger row.
- [ ] Raw events append-only, terminal status tepat sekali, dan backup checksum ada.
- [ ] Final counters, topology/role, model/gate/command audit tersedia sesuai eligibility.
- [ ] Secrets dan private identifiers tidak masuk public repository.

**Reproduction**

- [ ] `reproduce.py` mempunyai explicit input/output/exit/dependency contract.
- [ ] Known-evidence positive dan negative fixtures lulus.
- [ ] Manifest digest dan every NDJSON record diverifikasi.
- [ ] Lifecycle dan aggregate reconciliation fail nonzero pada inconsistency.
- [ ] Calibration/held-out, horizon, metric, interval, seeds, dan exclusions frozen.
- [ ] Clean-clone second-person reproduction menghasilkan output/verdict yang sama.
- [ ] Derived output mencatat input digests dan analysis identity tanpa overwrite raw.

**Communication**

- [ ] Positive, negative, inconclusive, dan non-evaluable dibedakan.
- [ ] Result table menyertakan uncertainty serta valid/invalid/interrupted counts.
- [ ] Safety claim dibuat per invariant; failure mempertahankan remote-off.
- [ ] Limitations membatasi external validity secara eksplisit.
- [ ] README/EXPERIMENTS hanya mengklaim apa yang evidence buktikan.
- [ ] Tidak ada fitur baru, post-hoc threshold, favorable replacement run, atau fabricated placeholder.

Checklist yang belum tercentang tetap terlihat. Week 6 selesai bukan ketika semua outcome positif, melainkan ketika evidence chain lengkap, failure behavior jujur, reproduction dapat diulang, dan claim tidak melampaui bukti.
